# More than a Hack 2027  Phase 1: Ball Possession

### A complete pipeline that reads every asset straight from Google Drive

**Task.** For each of the 199,319 frames of the *tactical* video, output `H` / `A` / `D` / `X`
(home possession, away possession, dead ball, abstain). Ranked on **spellF1** (possessions matched by
temporal IoU), tie-break **macroF1**.

**What this notebook does**

| § | what |
|---|---|
| 0 | Colab/Drive setup: mount Drive, **find the match bundle and the repo on Drive**, resolve paths |
| 1 | analysis of the repo: frame contract, graded frames, public ground truth, scorer, baselines  runs with no match data |
| 2 | load the bundle from Drive (tracking, Smart Tagging, homography) and verify alignment against the repo's own committed baseline |
| 3 | features, self-supervised models, evidence, decoder → `frame,label` CSV, validated |
| 4 | evaluation against *everything labelled that exists*, plus a stress test for the ball-visibility problem |
| 5 | the three deliverables `SUBMISSION.md` asks for: validated CSV, technical document, reproducibility manifest + contract checklist |

**Run it:** open in Colab, `Runtime → Run all`. The only thing you must provide is the match bundle
somewhere in your Drive (any folder, any depth). Everything else  the repo, the public ground truth,
the baselines, the scorer  is fetched and read automatically.

**Honest limits, up front**

* The only labels any participant has are the **5 public minutes (17 possessions)**. The full-match ground truth is private, so the number that
  actually ranks you cannot be computed here  §4 gives the best available substitutes and says how far to trust each.
* This notebook never trains on the public labels. They are used for *evaluation*, and for selecting the four constants that only labels can set (the dead-ball displacement cue, on its 1,469 dead frames; the tag weight; the isolation radius; the age-hazard gamma - all four listed in the document's section 6) - never for fitting a model, and never to pick the resolution. (Allowed by RULES.md, but 17 possessions would overfit instantly.)
* Do not upload the data, or extracts of it, to any hosted LLM (FAQ.md / T&Cs). Nothing here does.
* The competition repository is treated as read-only apart from `evaluation/results/<TEAM_NAME>.csv`
  (git-ignored, and where the scorer reads submissions). The data stays on Drive; deliverables are
  written to a work folder on your own Drive, never back into the repository.

## 0 · Setup  Drive mount, data discovery, repo resolution

Everything is configured here, in one place. Defaults are what you want for Colab; every one of them
can be overridden with an environment variable, which is also how you run this notebook headless.

| setting | env var | default |
|---|---|---|
| `TEAM_NAME` | `MTAH_TEAM` | `BAU` (names the CSV and the technical document) |
| `DRIVE_ROOT` | `MTAH_DRIVE_ROOT` | `/content/drive/MyDrive` when Drive is mounted |
| `DATA_DIR` | `MTAH_DATA_DIR` | auto-discovered on Drive (the repo's own convention) |
| `WORK_DIR` | `MTAH_WORK_DIR` | `<Drive>/mtah2027_work`  deliverables land here too |
| `REPO_DIR` | `MTAH_REPO_DIR` | auto (local → Drive → clone) |
| `MOUNT_DRIVE` | `MTAH_MOUNT_DRIVE` | `1` |
| `BOOTSTRAP_N` | `MTAH_BOOTSTRAP` | `1000` |
| `STRESS_DRAWS` | `MTAH_STRESS_DRAWS` | `10` |
| `STRICT_PARITY` | `MTAH_STRICT_PARITY` | `1` (stop if the parse does not reproduce the repo baseline) |

**The bundle layout the resolver looks for** (names exactly as DATA.md ships them; the folder that
*contains* `tactical/` is the data root):

```
<any folder on your Drive>/
├─ tactical/
│  ├─ BAR-ATH_tactical.mp4
│  ├─ BAR-ATH_tactical_atd.xml
│  ├─ BAR-ATH_tactical_atd.txt
│  ├─ BAR-ATH_tactical_homography.csv
│  └─ BAR-ATH_tactical_smart_tagging.xml
└─ broadcast/            (optional)
   ├─ BAR-ATH_broadcast.mp4
   ├─ BAR-ATH_broadcast_atd.xml
   ├─ BAR-ATH_broadcast_atd.txt
   └─ BAR-ATH_broadcast_homography.csv
```

A flat folder (all files together, no `tactical/` subfolder) and a **zip** on Drive are handled too 
the notebook stages them into scratch space so `games.asset_paths()` works unchanged.

In [ ]:
import os, sys, json, re, math, time, hashlib, itertools, difflib, subprocess, tempfile, zipfile, shutil, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive.*")

def env_str(name, default=None):
    v = os.environ.get(name)
    return default if v is None or v == "" else v

def env_flag(name, default):
    v = os.environ.get(name)
    if v is None or v == "":
        return bool(default)
    return v.strip().lower() in {"1", "true", "yes", "on"}

# ------------------------------ USER SETTINGS ------------------------------
TEAM_NAME      = env_str("MTAH_TEAM", "BAU")        # submission -> evaluation/results/<TEAM_NAME>.csv
DRIVE_ROOT     = env_str("MTAH_DRIVE_ROOT", None)   # None = auto-detect / mount
DATA_DIR       = env_str("MTAH_DATA_DIR", None)     # None = search Drive for the bundle
WORK_DIR       = env_str("MTAH_WORK_DIR", None)     # None = <Drive>/mtah2027_work
REPO_DIR       = env_str("MTAH_REPO_DIR", None)     # None = find locally, then clone
REPO_URL       = env_str("MTAH_REPO_URL", "https://github.com/metrica-sports/more-than-a-hack-2027.git")
MOUNT_DRIVE    = env_flag("MTAH_MOUNT_DRIVE", True)
SEARCH_SECONDS = int(env_str("MTAH_SEARCH_SECONDS", "300"))   # budget for each Drive search
BOOTSTRAP_N    = int(env_str("MTAH_BOOTSTRAP", "1000"))       # bootstrap draws for CIs
STRESS_DRAWS   = int(env_str("MTAH_STRESS_DRAWS", "10"))      # 4B Monte-Carlo repeats
STRICT_PARITY  = env_flag("MTAH_STRICT_PARITY", True)         # stop if the parse != repo baseline
COPY_ATD_LOCAL = env_flag("MTAH_COPY_ATD_LOCAL", True)        # stage the big text file in scratch
USE_CACHE      = env_flag("MTAH_USE_CACHE", True)             # parsed-tracking npz cache
AUTO_CLONE     = True
# ---------------------------------------------------------------------------

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

COLAB = in_colab()

SKIP_DIRS = {".Trash", ".Trashes", ".git", "node_modules", "__pycache__", ".ipynb_checkpoints",
             "$RECYCLE.BIN", "System Volume Information", ".cache", ".config"}

def search_tree(root, accept, budget=None, label="search", max_depth=7):
    """Depth-first search for the first path accepted by `accept`, with a hard time budget."""
    budget = SEARCH_SECONDS if budget is None else budget
    t0 = time.time()
    seen = 0
    stack = [(str(root), 0)]
    while stack:
        folder, depth = stack.pop()
        if time.time() - t0 > budget:
            print("  [%s] gave up after %.0fs / %d folders" % (label, budget, seen))
            return None
        try:
            entries = list(os.scandir(folder))
        except OSError:
            continue
        seen += 1
        if seen % 250 == 0:
            print("  [%s] %d folders, %.0fs ..." % (label, seen, time.time() - t0))
        for entry in entries:
            try:
                if entry.is_file(follow_symlinks=False) and accept(entry.path):
                    return Path(entry.path)
                if (entry.is_dir(follow_symlinks=False) and depth < max_depth
                        and entry.name not in SKIP_DIRS and not entry.name.startswith(".")):
                    stack.append((entry.path, depth + 1))
            except OSError:
                continue
    return None

def mount_drive():
    """Mount Drive in Colab (idempotent) and return its MyDrive root, or a local Google Drive path."""
    if not MOUNT_DRIVE:
        return None
    if COLAB:
        from google.colab import drive
        try:
            drive.mount("/content/drive")
        except Exception as exc:                 # already mounted is not an error
            print("drive.mount:", exc)
        for cand in (Path("/content/drive/MyDrive"), Path("/content/drive/My Drive")):
            if cand.is_dir():
                return cand
        return Path("/content/drive")
    for cand in (Path.home() / "Google Drive" / "MyDrive",
                 Path.home() / "Google Drive",
                 Path("/mnt/drive/MyDrive"),
                 Path("/Volumes/GoogleDrive/My Drive")):
        if cand.is_dir():
            return cand
    return None

DRIVE_ROOT = Path(DRIVE_ROOT).expanduser() if DRIVE_ROOT else mount_drive()
if DRIVE_ROOT is not None and not DRIVE_ROOT.is_dir():
    print("WARNING: Drive root does not exist, continuing without Drive:", DRIVE_ROOT)
    DRIVE_ROOT = None

# Scratch space: never on Drive, never in the repo, wiped with the Colab runtime.
CACHE_DIR = Path("/content/mtah2027_cache") if COLAB else Path(tempfile.gettempdir()) / "mtah2027_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if WORK_DIR:
    WORK_DIR = Path(WORK_DIR).expanduser()
elif DRIVE_ROOT is not None:
    WORK_DIR = DRIVE_ROOT / "mtah2027_work"      # persistent: cloned repo + deliverables
else:
    WORK_DIR = CACHE_DIR / "work"
try:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
except OSError as exc:
    print("WARNING: cannot write to %s (%s); falling back to scratch" % (WORK_DIR, exc))
    WORK_DIR = CACHE_DIR / "work"
    WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORK_DIR / "submission"                # CSV + technical document + manifest, for upload
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("colab        :", COLAB)
print("Drive root   :", DRIVE_ROOT if DRIVE_ROOT is not None else "not available (local run)")
print("work dir     :", WORK_DIR)
print("deliverables :", OUT_DIR)
print("scratch      :", CACHE_DIR)

### 0.1 The repository

`games.py`, `evaluation/` and `starter/` are the contract: constants, the metric library, the scorer,
the public ground truth, the four baselines. They are read from, in this order:

1. `MTAH_REPO_DIR` / the current directory / its parents (a local clone),
2. a clone already sitting in `WORK_DIR` on Drive (so it survives a runtime restart),
3. a fresh `git clone` into `WORK_DIR` on Drive, then into scratch,
4. a search of Drive for a folder containing `games.py` + `evaluation/possession_metrics.py`.

Whichever it finds is printed, so there is never any doubt which code produced the numbers.

In [ ]:
def inside(path, root):
    """True if `path` is `root` or below it (component-wise: 'mtah' must not match 'mtah2027_cache')."""
    try:
        Path(path).resolve().relative_to(Path(root).resolve())
        return True
    except (ValueError, OSError):
        return False

def looks_like_repo(p):
    p = Path(p)
    return (p / "games.py").is_file() and (p / "evaluation" / "possession_metrics.py").is_file()

def _describe_location(path):
    if DRIVE_ROOT is not None and inside(path, DRIVE_ROOT):
        return "google drive"
    return "local"

def resolve_repo():
    candidates = []
    if REPO_DIR:
        candidates.append(Path(REPO_DIR).expanduser())
    here = Path.cwd().resolve()
    candidates += [here, *here.parents,
                   here / "more-than-a-hack-2027",
                   WORK_DIR / "more-than-a-hack-2027",
                   CACHE_DIR / "more-than-a-hack-2027"]
    for cand in candidates:
        if looks_like_repo(cand):
            return cand.resolve(), _describe_location(cand.resolve())

    for target in (WORK_DIR / "more-than-a-hack-2027", CACHE_DIR / "more-than-a-hack-2027"):
        if not AUTO_CLONE:
            continue
        print("cloning", REPO_URL, "->", target)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                           check=True, capture_output=True, text=True)
        except Exception as exc:
            print("  clone failed:", getattr(exc, "stderr", exc))
            continue
        if looks_like_repo(target):
            return target.resolve(), _describe_location(target.resolve())

    if DRIVE_ROOT is not None:                       # last resort: the repo itself may be on Drive
        print("searching Drive for a copy of the repository ...")
        hit = search_tree(DRIVE_ROOT, lambda p: os.path.basename(p) == "possession_metrics.py"
                          and (Path(p).parent.parent / "games.py").is_file(),
                          budget=SEARCH_SECONDS, label="repo", max_depth=6)
        if hit is not None:
            return Path(hit).parent.parent.resolve(), "google drive"
    return None, None

REPO, REPO_SOURCE = resolve_repo()
assert REPO is not None, (
    "Could not find or clone metrica-sports/more-than-a-hack-2027.\n"
    "Set MTAH_REPO_DIR to a clone, or make sure git is available so it can clone.")

for sub in ("", "evaluation", "starter"):
    path = str(REPO / sub)
    if path not in sys.path:
        sys.path.insert(0, path)

import numpy as np
import pandas as pd

pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 4)

import games
import possession_metrics as pm
import baseline as repo_baseline            # starter/baseline.py (team mapping + reference baseline)
import load_smart_tagging, load_homography

try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception as exc:                    # plotting is optional
    HAVE_PLT = False
    print("matplotlib unavailable:", exc)

print("repo   :", REPO, "(%s)" % REPO_SOURCE)
print("python :", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)
print("game   :", games.GAME, "| frames", games.GAMES[games.GAME]["n_video_frames"],
      "| fps", games.GAMES[games.GAME]["fps"])
REPO_REV = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip() or "unknown"
print("commit :", REPO_REV)

### 0.2 The match bundle, read from Google Drive

The repository deliberately hard-codes no filename: `games.asset_paths()` builds every path from
`games.DATA_DIR`. So the whole job here is to point `games.DATA_DIR` at the right folder  on Drive 
and let the repo's own loaders do the rest. No copying into `repo/data`, no `MTAH_DATA_DIR` export
needed.

Five things are searched for, in order:

1. `MTAH_DATA_DIR`, if you set it (a bundle root, a `tactical/` folder, or the raw txt itself),
2. the Drive tree for `*_tactical_atd.txt` (depth-limited, time-limited, junk directories pruned),
3. a zip on Drive whose name looks like the bundle, extracted once into scratch,
4. a flat folder with the assets side by side, staged into scratch with symlinks,
5. nothing  in which case §0–§1 still run and §2 explains what to upload.

The search result is printed with **where** each file is, so "is this really reading Drive?" is
answered by the output rather than by trust.

> **Shared-with-me folders are not visible to `drive.mount()`.** If the bundle lives in a folder that
> was shared with you rather than created by you, either move/copy it into *My Drive* or upload the zip
> there  both are found automatically. Colab can only see `MyDrive`.

In [ ]:
def is_tactical_atd(path):
    return os.path.basename(path).lower().endswith("_tactical_atd.txt")

def stage_layout(src_dir):
    """Flat folder -> scratch/tactical|broadcast symlinks, so games.asset_paths() works unchanged."""
    stage = CACHE_DIR / "staging"
    for sub in ("tactical", "broadcast"):
        (stage / sub).mkdir(parents=True, exist_ok=True)
    for f in sorted(Path(src_dir).iterdir()):
        if not f.is_file():
            continue
        low = f.name.lower()
        sub = "tactical" if "_tactical" in low else ("broadcast" if "_broadcast" in low else None)
        if sub is None:
            continue
        dest = stage / sub / f.name
        if dest.is_symlink() or dest.exists():
            continue
        try:
            os.symlink(str(f.resolve()), str(dest))
        except OSError:                       # Windows without developer mode, or a FUSE mount
            shutil.copyfile(str(f), str(dest))
    return stage

def root_from_atd(atd_path):
    parent = Path(atd_path).parent
    if parent.name.lower() == "tactical":
        return parent.parent
    return stage_layout(parent)

DATA_ROOT, DATA_SOURCE, DATA_FOUND_AT = None, "not found", None

if DATA_DIR:                                                     # 1) explicit
    p = Path(DATA_DIR).expanduser()
    if not p.is_absolute() and DRIVE_ROOT is not None:
        p = DRIVE_ROOT / p
    if p.is_file() and is_tactical_atd(p):
        DATA_ROOT, DATA_SOURCE, DATA_FOUND_AT = root_from_atd(p), "MTAH_DATA_DIR", p.parent
    elif p.is_dir():
        hit = search_tree(p, is_tactical_atd, budget=90, label="MTAH_DATA_DIR", max_depth=5)
        if hit is not None:
            DATA_ROOT, DATA_SOURCE, DATA_FOUND_AT = root_from_atd(hit), "MTAH_DATA_DIR", hit.parent
        else:
            print("  no *_tactical_atd.txt under", p)

if DATA_ROOT is None and DRIVE_ROOT is not None:                  # 2) Drive search
    print("searching Google Drive for the match bundle (this is the slow part, once) ...")
    hit = search_tree(DRIVE_ROOT, is_tactical_atd, label="drive")
    if hit is not None:
        DATA_ROOT, DATA_SOURCE, DATA_FOUND_AT = root_from_atd(hit), "google drive", hit.parent

if DATA_ROOT is None and DRIVE_ROOT is not None:                  # 3) a zip on Drive
    print("no unpacked bundle found - looking for a zip on Drive ...")
    hit = search_tree(DRIVE_ROOT,
                      lambda p: p.lower().endswith(".zip") and any(
                          key in os.path.basename(p).lower()
                          for key in ("mtah", "hack", "bar-ath", "tactical")),
                      label="drive zip")
    if hit is not None:
        target = CACHE_DIR / "bundle"
        shutil.rmtree(target, ignore_errors=True)
        target.mkdir(parents=True, exist_ok=True)
        print("  extracting", hit, "->", target)
        with zipfile.ZipFile(str(hit)) as archive:
            archive.extractall(str(target))
        inner = search_tree(target, is_tactical_atd, budget=180, label="extracted", max_depth=6)
        if inner is not None:
            DATA_ROOT, DATA_SOURCE = root_from_atd(inner), "google drive (zip, extracted to scratch)"
            DATA_FOUND_AT = inner.parent

PATHS, NEED, DATA_OK = {}, ["atd_meta", "atd_raw", "homography", "smart_tagging"], False
if DATA_ROOT is not None:
    os.environ["MTAH_DATA_DIR"] = str(DATA_ROOT)
    games.DATA_DIR = str(DATA_ROOT)                 # the one line every loader reads
    PATHS = games.asset_paths()
    missing = [PATHS[name] for name in NEED if not os.path.isfile(PATHS[name])]
    DATA_OK = not missing

    rows = []
    for name in list(NEED) + ["video"]:
        path = PATHS[name]
        present = os.path.isfile(path)
        rows.append(dict(asset=name, present=present,
                         size_MB=round(os.path.getsize(path) / 1e6, 1) if present else np.nan,
                         on_drive=bool(DRIVE_ROOT is not None and inside(path, DRIVE_ROOT)),
                         path=path))
    print("\ndata root :", DATA_ROOT)
    print("data from :", DATA_SOURCE)
    if DATA_FOUND_AT is not None and Path(DATA_ROOT).name == "staging":
        print("bundle at :", DATA_FOUND_AT, "(flat folder - staged into scratch so games.asset_paths() works)")
    display(pd.DataFrame(rows).set_index("asset"))

    broadcast = games.asset_paths(source="broadcast")
    print("broadcast (optional, never used below):",
          {name: os.path.isfile(path) for name, path in broadcast.items()})
    if not DATA_OK:
        print("\nMISSING:")
        for path in missing:
            print("  " + path)
        print("Upload the bundle to Drive (a folder holding tactical/ and broadcast/), or set MTAH_DATA_DIR.")
else:
    print("""
DATA NOT FOUND - sections 0 and 1 still run; section 2 stops with a readable message.

Put the downloaded bundle anywhere in your Google Drive, keeping the folder that contains
tactical/ and broadcast/, for example:

    MyDrive/FCB Hackathon/data/tactical/BAR-ATH_tactical_atd.txt
    MyDrive/FCB Hackathon/data/broadcast/...

or set DATA_DIR (env MTAH_DATA_DIR) to that folder. A zip is also accepted.

Colab's drive.mount() only exposes My Drive: a folder that was merely shared with you will not be
found, so add it to My Drive first.
""")
    if DRIVE_ROOT is not None:
        try:
            names = sorted(e.name for e in os.scandir(DRIVE_ROOT) if e.is_dir())
            print("top-level folders in your Drive (set DATA_DIR to the one holding the bundle):")
            for name in names[:40]:
                print("   " + name)
        except OSError:
            pass

## 1 · Analysis of the repo (no match data needed)

### 1.1 The frame contract and which frames are graded

Everything is indexed by **tactical video frame**, 0…199318. Tracking `frame_id` counts from 1
(`video_frame = frame_id − 1`, verified in `starter/load_atd.py`); Smart Tagging is in seconds
(`round(s·25)`); homography is already 0-based. The scorer only looks at the two halves.

In [ ]:
GAME = games.GAME
N    = games.GAMES[GAME]["n_video_frames"]
FPS  = games.GAMES[GAME]["fps"]
L_M, W_M = games.GAMES[GAME]["pitch"]                    # metres (105.3 x 68.0)
BLOCKS = games.half_video_blocks(GAME)                   # [(1, first, last), (2, first, last)] inclusive
RUNS   = [(a, b + 1) for _, a, b in BLOCKS]              # half-open, same convention as pm.valid_runs
GRADED = np.zeros(N, bool); HALF = np.zeros(N, np.int8)
for h, a, b in BLOCKS:
    GRADED[a:b + 1] = True; HALF[a:b + 1] = h
KICKOFF = load_smart_tagging.KICKOFF_VIDEO_FRAMES        # {'1st Half': 11185, '2nd Half': 113742}
KICK1, KICK2 = KICKOFF["1st Half"], KICKOFF["2nd Half"]
HALF2_START = BLOCKS[1][1]

print("frames %d  fps %d  pitch %.1f x %.1f m" % (N, FPS, L_M, W_M))
for h, a, b in BLOCKS:
    print("half %d: video frames %d..%d  (%d frames, %.1f min)" % (h, a, b, b - a + 1, (b - a + 1) / FPS / 60))
print("graded %d / %d  (%.1f%%)" % (GRADED.sum(), N, 100 * GRADED.mean()))
print("TRAP: half 2 is graded from %d but kicks off at %d -> %d graded frames that are all D (FAQ.md)"
      % (HALF2_START, KICK2, KICK2 - HALF2_START))
assert KICK2 > HALF2_START, "the tagged 2nd-half kickoff is not after the 2nd-half start - check the tag time base"

# Graded but guaranteed dead by rule: the run before the 2nd-half kickoff. Rates must not divide by it.
PLAY_FRAMES = int(GRADED.sum() - (KICK2 - HALF2_START))

### 1.2 The public ground truth  the only labels that exist

`evaluation/gt/BAR-ATH_possession_gt_public.csv`: full-length, real labels only on video frames 11185…18684 (5 min), `X` elsewhere.

In [ ]:
GT_DIR = REPO / "evaluation" / "gt"
GT_CANDIDATES = [GT_DIR / "BAR-ATH_possession_gt_public.csv"] + sorted(GT_DIR.glob("*public*.csv"))
GT_PATH = next((p for p in GT_CANDIDATES if p.is_file()), None)
assert GT_PATH is not None, "no public ground-truth CSV in %s - is this the right repo?" % GT_DIR
gt = pm.label_codes(pm.read_label_csv(GT_PATH))
print("ground truth:", GT_PATH.name)
GT_RUNS = pm.valid_runs(gt)
WIN_LO, WIN_HI = GT_RUNS[0]                                   # half-open window
print("public window:", WIN_LO, "..", WIN_HI - 1, "(%.1f min)" % ((WIN_HI - WIN_LO) / FPS / 60))

NAMES = "HADX"
seg = pd.DataFrame([(NAMES[k], a, b, (b - a) / FPS, (a - WIN_LO) / FPS) for a, b, k in pm.segments(gt, GT_RUNS)],
                   columns=["label", "start", "stop", "dur_s", "t_in_window_s"])
display(seg.round(1))
poss = seg[seg.label.isin(["H", "A"])]
print("%d possessions | mean %.1fs | median %.1fs | shortest %.1fs | longest %.1fs | %d are shorter than 4 s"
      % (len(poss), poss.dur_s.mean(), poss.dur_s.median(), poss.dur_s.min(), poss.dur_s.max(), (poss.dur_s < 4).sum()))
d = seg[seg.label == "D"]
print("%d dead-ball stretches, %.1f-%.1fs, D share of window %.1f%%"
      % (len(d), d.dur_s.min(), d.dur_s.max(), 100 * d.dur_s.sum() / seg.dur_s.sum()))

### 1.3 The scorer, reproduced and instrumented
`pm.evaluate` is the same function `run_evaluation.py` calls. Helpers below turn it into tables we reuse everywhere.

Key mechanics worth internalising (all from `possession_metrics.py`):

* **Possession = maximal run of one team with ball alive.** `H H D D H H` is *two* possessions  every stoppage splits a possession, so dead-ball timing matters as much as team calls.
* **spellF1** = F1 of one-to-one matched possessions, averaged over IoU 0.3…0.7. A one-frame offset is free; 1 s late costs ~0.35.
* **`X` is forward-filled** (per half) before spellF1  so an abstention means "repeat my last answer", cheap inside a possession, expensive across a hand-over. `macroF1` counts `X` as a miss.

In [ ]:
def score_one(gt_codes, pred_codes, bootstrap_n=0):
    return pm.evaluate(gt_codes, pred_codes, bootstrap_n=bootstrap_n)

def row_from_result(name, r):
    pc = r["per_class"]
    return dict(submission=name, spellF1=r["spell_f1_mean"], spell_lo=r["spell_bootstrap_low"], spell_hi=r["spell_bootstrap_high"],
                macroF1=r["macro_f1"], poss=r["n_pred_spells"], gt_poss=r["n_gt_spells"], dur_s=r["mean_spell_duration_s"],
                F1_H=pc["H"]["f1"], F1_A=pc["A"]["f1"], F1_D=pc["D"]["f1"], alive_dead_F1=r["alive_dead_f1"],
                team_acc=r["team_acc_both_alive"], transF1_25=r["transitions"][25]["f1"], frag=r["fragmentation"])

def score_table(preds, gt_codes, bootstrap_n=0):
    rows = [row_from_result(n, score_one(gt_codes, c, bootstrap_n)) for n, c in preds.items()]
    return pd.DataFrame(rows).sort_values(["spellF1", "macroF1"], ascending=False).reset_index(drop=True)

def possession_report(gt_codes, pred_codes, thr=0.5):
    # each true possession with the best-overlapping predicted possession of the same team
    runs = pm.valid_runs(gt_codes)
    filled = pm.fill_abstentions(pred_codes, runs)
    gs, ps = pm.possession_spells(gt_codes, runs), pm.possession_spells(filled, runs)
    ov = pm.spell_overlaps(gs, ps); hit = pm.matched_gt_flags(ov, len(gs), thr)
    best = {}
    for iou, i, j in ov:
        best.setdefault(i, (iou, j))
    rows = []
    for i, (a, b, k) in enumerate(gs):
        iou, j = best.get(i, (0.0, None))
        rows.append(dict(team=NAMES[k], start=a, dur_s=(b - a) / FPS, best_iou=iou, matched=bool(hit[i]),
                         pred_start=ps[j][0] if j is not None else None,
                         pred_dur_s=(ps[j][1] - ps[j][0]) / FPS if j is not None else None))
    return pd.DataFrame(rows)

def run_bounds(mask):
    d = np.diff(np.concatenate(([0], np.asarray(mask).astype(np.int8), [0])))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)

def load_codes(path):
    return pm.label_codes(pm.read_label_csv(path))

BASELINES = {p.stem: load_codes(p) for p in sorted((REPO / "evaluation" / "results").glob("baseline_*.csv"))}
assert BASELINES, "no baseline_*.csv in evaluation/results - is this the right repo?"
print("baselines found:", list(BASELINES))

### 1.4 Baselines on the public window  and why the board isn't a leaderboard

In [ ]:
base_tab = score_table(BASELINES, gt, BOOTSTRAP_N)
display(base_tab.round(3))
print("Same algorithm four times, only the smoothing window changes. macroF1 barely moves; spellF1 moves ~11x.")
print("Published FULL-MATCH spellF1 (EVALUATION.md): raw 0.0486 | smooth25 0.1996 | smooth50 0.2433 | smooth125 0.2242")
print("-> on the public window smooth125 'wins' but on the full match it is worse than smooth50. 17 possessions cannot rank anything.")

### 1.5 What the shipped baseline files reveal about the *whole match* (still no data needed)
`baseline_raw.csv` labels `D` exactly where the ball is not tracked, so it doubles as a **ball-visibility mask for all 199,319 frames**.

In [ ]:
raw = BASELINES["baseline_raw"]
base_dm = (raw == pm.D)          # what the baseline calls dead: no ball OR nobody near one

# This cell can only speak for the baseline: the ATD is not parsed yet in this section, so the ball's own
# channel is measured in 2.2, right after VIS exists. Keeping the two apart is the point - the baseline's D
# is "no ball OR nobody near it", which is a different and much larger set than "the ball is missing".
print("the baseline calls D on %.1f%% of graded frames and %.1f%% of the public window"
      % (100 * base_dm[GRADED].mean(), 100 * base_dm[WIN_LO:WIN_HI].mean()))

print("\nfull-match possession counts of the shipped baselines:")
for name, codes in BASELINES.items():
    spells = pm.possession_spells(codes, RUNS)
    lengths = np.array([b - a for a, b, _ in spells]) / FPS
    print("  %-20s %5d possessions, mean %5.1fs, median %4.1fs"
          % (name, len(spells), lengths.mean(), np.median(lengths)))

## 2 · Load the gated data bundle from Drive

The files are wherever `games.DATA_DIR` now points  the Drive folder found above. If they are
missing this section stops with a readable message; **everything above still ran**.

**Parsing choices.** We read the raw EPTS text ourselves with numpy (no kloppy needed): lines look like
`frame_id:slot;slot;…:ball`, each group `x,y,speed`, coordinates normalised 0–1, and a whole missing
group written `N` (the supplied export) or `NaN` (the generic EPTS spelling)  both are handled, and
CRLF is accepted. A chunk is cleaned with two regex passes (a missing group becomes the right number
of NaNs  players and ball can have different field counts), separators become spaces, and the chunk
is parsed in one `numpy` call; anything unusual falls back to a slow line-by-line parser, so a format
surprise degrades speed, never correctness.

We then **prove the parser, team mapping and frame alignment are right** by reproducing the repo's
committed `baseline_raw.csv` from our arrays  that file was produced from the same ATD by
`starter/baseline.py`.

In [ ]:
if not DATA_OK:
    raise RuntimeError(
        "Match data not found - see section 0.2. Upload the bundle to Google Drive and re-run,\n"
        "or set MTAH_DATA_DIR to the folder that contains tactical/.")

import xml.etree.ElementTree as ET

# Stage the (large) tracking text in scratch: Drive is a network mount, and we read this file twice.
ATD_DRIVE_PATH = PATHS["atd_raw"]
ATD_PARSE_PATH = ATD_DRIVE_PATH
if COPY_ATD_LOCAL:
    local_atd = CACHE_DIR / "atd" / Path(ATD_DRIVE_PATH).name
    local_atd.parent.mkdir(parents=True, exist_ok=True)
    if (not local_atd.is_file()) or local_atd.stat().st_size != os.path.getsize(ATD_DRIVE_PATH):
        print("copying the tracking text into scratch (%s) ..." % local_atd)
        t0 = time.time()
        shutil.copyfile(ATD_DRIVE_PATH, local_atd)
        print("  %.0fs, %.1f MB" % (time.time() - t0, local_atd.stat().st_size / 1e6))
    else:
        print("scratch copy already present:", local_atd)
    ATD_PARSE_PATH = str(local_atd)

def describe_metadata(meta_path, max_lines=40):
    # What channels does the ATD declare? (Does the ball carry anything beyond x,y,speed?)
    root = ET.parse(meta_path).getroot()
    counts = {}
    for el in root.iter():
        tag = el.tag.split("}")[-1]; counts[tag] = counts.get(tag, 0) + 1
    print("element counts:", {k: v for k, v in sorted(counts.items(), key=lambda kv: -kv[1])[:14]})
    shown = 0
    for el in root.iter():
        tag = el.tag.split("}")[-1]
        if ("Channel" in tag or "Spec" in tag or "Unit" in tag) and shown < max_lines:
            print("  ", tag, dict(el.attrib), (el.text or "").strip()[:60]); shown += 1

describe_metadata(PATHS["atd_meta"])
with open(ATD_PARSE_PATH) as fh:
    last = None
    for _ in range(3):
        last = fh.readline()
print("\nsample raw line (truncated):", last[:230].rstrip(), "...", last[-90:].rstrip())

In [ ]:
_TRANS = str.maketrans({":": " ", ";": " ", ",": " ", "\r": " "})
# A whole missing group is one token: `N` in the delivered export, `NaN` in the generic EPTS
# convention (starter/baseline.py tests only the first character for exactly that reason). Both
# are accepted, case-insensitively, and expanded to the right number of NaNs so one numpy call can
# read the chunk. The ball is always the last group on the line, so the two cases are
# unambiguous: a bare token closed by end of line is the ball, one closed by ; or : is a player
# slot. They may have different field counts (the ball is often x,y; players x,y,speed).
_NAN_BALL = re.compile(r"(?<=:)(?:NaN|N)(?=[\r\n]|$)", re.IGNORECASE)
_NAN_PLAYER = re.compile(r"(?<=[:;])(?:NaN|N)(?=[;:])", re.IGNORECASE)

def _probe_layout(path, stride=997):
    n_slots = n_pf = n_bf = 0
    with open(path) as fh:
        for i, line in enumerate(fh):
            if i % stride:
                continue
            parts = line.rstrip("\n").split(":")
            if len(parts) != 3:
                continue
            groups = parts[1].split(";")
            n_slots = max(n_slots, len(groups))
            n_pf = max([n_pf] + [len(g.split(",")) for g in groups])
            n_bf = max(n_bf, len(parts[2].split(",")))
    return n_slots, n_pf, n_bf

def _fast_parse(lines, n_slots, n_pf, n_bf, width):
    text = "".join(lines)
    if not text.endswith("\n"):
        text += "\n"
    text = _NAN_BALL.sub(",".join(["nan"] * n_bf), text)
    text = _NAN_PLAYER.sub(",".join(["nan"] * n_pf), text).translate(_TRANS)
    try:
        a = np.fromstring(text, sep=" ", dtype=np.float64)
    except Exception:
        return None
    if a.size != len(lines) * width:
        return None
    return a.reshape(len(lines), width)

def _parse_slow(lines, n_slots, n_pf, n_bf, width):
    out = np.full((len(lines), width), np.nan)
    for r, line in enumerate(lines):
        parts = line.rstrip("\r\n").split(":")
        if len(parts) != 3:
            continue
        out[r, 0] = float(parts[0])
        for s, g in enumerate(parts[1].split(";")[:n_slots]):
            if g and g.strip().lower() not in ("n", "nan"):
                v = g.split(",")
                if len(v) >= n_pf:
                    out[r, 1 + s * n_pf: 1 + (s + 1) * n_pf] = [float(x) for x in v[:n_pf]]
        g = parts[2]
        if g and g.strip().lower() not in ("n", "nan"):
            v = g.split(",")
            if len(v) >= n_bf:
                out[r, 1 + n_slots * n_pf:] = [float(x) for x in v[:n_bf]]
    return out

def parse_atd(path, n_frames, chunk=5000, verbose=True):
    n_slots, n_pf, n_bf = _probe_layout(path)
    width = 1 + n_slots * n_pf + n_bf
    P = np.full((n_frames, n_slots, n_pf), np.nan, np.float32)
    B = np.full((n_frames, n_bf), np.nan, np.float32)
    seen = np.zeros(n_frames, bool)
    slow = fast = chunks = 0
    t0 = time.time()
    with open(path) as fh:
        while True:
            lines = list(itertools.islice(fh, chunk))
            if not lines:
                break
            arr = _fast_parse(lines, n_slots, n_pf, n_bf, width)
            if arr is None:
                arr = _parse_slow(lines, n_slots, n_pf, n_bf, width); slow += 1
            else:
                fast += 1
            ids = np.where(np.isnan(arr[:, 0]), -1.0, arr[:, 0] - 1.0)
            ids = np.rint(ids).astype(np.int64)
            ok = (ids >= 0) & (ids < n_frames)
            ids, arr = ids[ok], arr[ok]
            P[ids] = arr[:, 1: 1 + n_slots * n_pf].reshape(-1, n_slots, n_pf)
            B[ids] = arr[:, 1 + n_slots * n_pf:]
            seen[ids] = True
            chunks += 1
            if verbose and chunks % 20 == 0:
                print("  %6d / %6d frames, %.0fs" % (min(n_frames, chunks * chunk), n_frames, time.time() - t0))
    return P, B, seen, dict(n_slots=n_slots, n_player_fields=n_pf, n_ball_fields=n_bf,
                            chunks_fast=fast, chunks_slow=slow)

CACHE_FILE = CACHE_DIR / "atd_tactical.npz"
src_mtime = os.path.getmtime(ATD_DRIVE_PATH)
t0 = time.time()
if USE_CACHE and CACHE_FILE.is_file() and CACHE_FILE.stat().st_mtime >= src_mtime:
    z = np.load(CACHE_FILE, allow_pickle=True)
    P_RAW, B_RAW, SEEN, LAYOUT = z["P"], z["B"], z["seen"], z["layout"].item()
    print("loaded the parsed tracking from cache (%.0fs)" % (time.time() - t0))
else:
    P_RAW, B_RAW, SEEN, LAYOUT = parse_atd(ATD_PARSE_PATH, N)
    if USE_CACHE:
        np.savez(CACHE_FILE, P=P_RAW, B=B_RAW, seen=SEEN, layout=np.array(LAYOUT, dtype=object))
    print("parsed in %.0fs and cached to %s" % (time.time() - t0, CACHE_FILE))

print("layout:", LAYOUT)
print("records present %d / %d  -> one record per video frame, ids 1..N: %s"
      % (SEEN.sum(), N, bool(SEEN.all())))
if LAYOUT["chunks_slow"]:
    print("NOTE: %d of %d chunks needed the tolerant line-by-line parser (mixed field counts); "
          "slower, same result. The parity check below is what proves it."% (
              LAYOUT["chunks_slow"], LAYOUT["chunks_slow"] + LAYOUT["chunks_fast"]))
if not SEEN.all():
    print("WARNING: gaps in the tracking record ids - do not invent a correction, check DATA.md alignment")
if LAYOUT["n_ball_fields"] > 3:
    print("NOTE: the ball has", LAYOUT["n_ball_fields"], "fields - inspect B_RAW[:, 3:], it may carry status/ownership")

In [ ]:
# ---- team membership of each track slot (first team in the metadata = home, the FIFA/EPTS convention) ----
def metadata_teams(meta_path):
    """The same rule as starter/baseline.py, but namespace-agnostic: first Team is home.

    Only used if the repository's own reader raises (an XML dialect with namespaced tags would make
    ElementTree's `iter("Team")` find nothing). The parity check below is what proves the mapping.
    """
    root = ET.parse(meta_path).getroot()
    def local(el):
        return el.tag.split("}")[-1]
    teams = [el for el in root.iter() if local(el) == "Team"]
    players = [el for el in root.iter() if local(el) == "Player"]
    if len(teams) != 2 or not players:
        raise ValueError("%s: expected 2 Teams and at least one Player, found %d and %d"
                         % (meta_path, len(teams), len(players)))
    home_id = teams[0].get("id")
    names = []
    for team in teams:
        names.append(next(((child.text or "").strip() for child in team if local(child) == "Name"), None))
    slots = np.array([0 if player.get("teamId") == home_id else 1 for player in players], np.int8)
    return slots, tuple(names)

try:
    TEAM_OF_SLOT = repo_baseline.team_of_each_track(PATHS["atd_meta"])       # 0 home / 1 away, per slot
    HOME_NAME, AWAY_NAME = repo_baseline.team_names(PATHS["atd_meta"])
    TEAM_MAP_SOURCE = "starter/baseline.py"
except Exception as exc:
    TEAM_OF_SLOT, (HOME_NAME, AWAY_NAME) = metadata_teams(PATHS["atd_meta"])
    TEAM_MAP_SOURCE = "namespace-agnostic fallback (%s)" % exc
TEAM_OF_SLOT = np.asarray(TEAM_OF_SLOT, np.int8)
HOME_NAME = HOME_NAME or "home"          # a metadata file without <Name> must not crash the run
AWAY_NAME = AWAY_NAME or "away"
assert len(TEAM_OF_SLOT) == P_RAW.shape[1], (len(TEAM_OF_SLOT), P_RAW.shape)
print("H = %s  (%d slots) | A = %s  (%d slots)   [team map: %s]"
      % (HOME_NAME, (TEAM_OF_SLOT == 0).sum(), AWAY_NAME, (TEAM_OF_SLOT == 1).sum(), TEAM_MAP_SOURCE))

PX = P_RAW[:, :, 0].astype(np.float64) * L_M          # metres
PY = P_RAW[:, :, 1].astype(np.float64) * W_M
PS = P_RAW[:, :, 2].astype(np.float64) if P_RAW.shape[2] > 2 else np.full(PX.shape, np.nan)   # m/s as delivered
BX = B_RAW[:, 0].astype(np.float64) * L_M
BY = B_RAW[:, 1].astype(np.float64) * W_M
VIS = ~np.isnan(BX)

n_obj = (~np.isnan(PX)).sum(1)
print("ball visible: all frames %.1f%% | graded %.1f%% | public window %.1f%%"
      % (100 * VIS.mean(), 100 * VIS[GRADED].mean(), 100 * VIS[WIN_LO:WIN_HI].mean()))
print("tracked objects per graded frame: mean %.1f  (>22 on %.1f%% of frames = duplicated tracks)"
      % (n_obj[GRADED].mean(), 100 * (n_obj[GRADED] > 22).mean()))
print("tracked objects at halftime (frame 100000): %d" % n_obj[100000])

# ---- nearest tracked player -> team code, chunked so it never needs a 200k x 46 float64 stack ----
def nearest_team_codes(PX, PY, BX, BY, team_of_slot, rows=None, chunk=50000):
    n = len(BX)
    lo, hi = (0, n) if rows is None else rows
    out = np.full(n, 2, np.int8)
    for a in range(lo, hi, chunk):
        b = min(hi, a + chunk)
        dx = PX[a:b] - BX[a:b, None]; dy = PY[a:b] - BY[a:b, None]
        d2 = dx * dx + dy * dy
        d2 = np.where(np.isnan(d2), np.inf, d2)
        idx = d2.argmin(1); best = d2[np.arange(b - a), idx]
        out[a:b] = np.where(np.isfinite(best), team_of_slot[idx], 2)
    return out

# ---- PARITY CHECK: reproduce the repo's committed baseline_raw.csv from our own arrays ----
mine = nearest_team_codes(PX, PY, BX, BY, TEAM_OF_SLOT)
agree = (mine == BASELINES["baseline_raw"])
mismatch = int((~agree).sum())
print("\nPARITY vs committed baseline_raw.csv: %.4f%% of all frames identical (%d differ)"
      % (100 * agree.mean(), mismatch))
# Exact equality is the expectation: the committed file is computed from the same text with the
# same pitch and the same tie-break (first nearest slot). A handful of frames can differ only
# where two players are equally distant and float32 storage rounds the wrong way, so allow a few
# -- a parser or alignment mistake shows up as thousands, not five.
PARITY = dict(agree=float(agree.mean()), mismatch=mismatch, compared_frames=int(len(agree)),
              baseline="baseline_raw.csv")
if mismatch <= 5:
    print("  OK - parser, team mapping and frame alignment reproduce the repo's own result.")
else:
    msg = ("MISMATCH - our arrays do not reproduce baseline_raw.csv. Check team order / frame offset / "
           "parser before trusting anything below.")
    if STRICT_PARITY:
        raise RuntimeError(msg + " (set MTAH_STRICT_PARITY=0 to continue anyway)")
    print("  " + msg)

# ---- the ball channel itself ----
ball_gap_mask = ~VIS
vis_graded = 1 - ball_gap_mask[GRADED].mean()
gap_list = []
for a, b in RUNS:
    starts, stops = run_bounds(ball_gap_mask[a:b])
    if len(starts):
        gap_list.append(stops - starts)
gaps = np.concatenate(gap_list) if gap_list else np.zeros(0, np.int64)

print("\nball visible: graded frames %.1f%%  |  public window %.1f%%   <- the window is the EASIEST part of the match"
      % (100 * (1 - ball_gap_mask[GRADED].mean()), 100 * (1 - ball_gap_mask[WIN_LO:WIN_HI].mean())))
if len(gaps):
    print("ball-missing gaps in graded frames: n=%d  median %.1fs  mean %.1fs  p90 %.0fs  max %.0fs"
          % (len(gaps), np.median(gaps) / FPS, gaps.mean() / FPS, np.percentile(gaps, 90) / FPS, gaps.max() / FPS))
    print("  missing frames in gaps > 20 s: %.1f%%  (%d gaps)"
          % (100 * gaps[gaps > 500].sum() / gaps.sum(), (gaps > 500).sum()))
    print("  => interpolating the ball across 'brief occlusions' is nearly useless: the gaps are long, so "
          "possession in them has to come from players + Smart Tagging.")

blk = []
for h, (a, b) in enumerate(RUNS, 1):
    blk += [(h, (s - a) / FPS / 60, 100 * (1 - ball_gap_mask[s:min(s + 7500, b)].mean())) for s in range(a, b, 7500)]
vis_blocks = pd.DataFrame(blk, columns=["half", "minute", "ball_visible_%"])
print("\nball visibility per 5-minute block:")
print(vis_blocks.pivot_table(index="half", columns=vis_blocks.groupby("half").cumcount(),
                             values="ball_visible_%").round(0).astype(int).to_string())

# the nearest-player rule, measured only where there is a ball to measure it against
raw = BASELINES["baseline_raw"]
in_win = (gt[WIN_LO:WIN_HI] != pm.D) & ~ball_gap_mask[WIN_LO:WIN_HI]
acc = (gt[WIN_LO:WIN_HI][in_win] == raw[WIN_LO:WIN_HI][in_win]).mean()
print("\nnearest-player-to-ball team == GT team on alive & ball-visible window frames: %.1f%%  "
      "(even a clean ball leaves ~22%% wrong)" % (100 * acc))
gtD = gt[WIN_LO:WIN_HI] == pm.D
print("GT dead-ball frames with the ball visible: %.1f%%  (so 'ball missing => D' is wrong most of the time)"
      % (100 * (1 - ball_gap_mask[WIN_LO:WIN_HI][gtD].mean())))

In [ ]:
# ---- Smart Tagging ----
TAGS = load_smart_tagging.load()
tag_report = load_smart_tagging.check_time_base(TAGS)
TAG_TIME_BASE_OK = bool(tag_report["agrees"])
print("%d tag instances | time base agrees with the real kickoffs: %s" % (len(TAGS), TAG_TIME_BASE_OK))
if not TAG_TIME_BASE_OK:
    for row in tag_report["halves"]:
        print("  ", row)
    print("WARNING: the tag clock is not aligned with the video, so tag evidence is disabled below.")
    print("         (RULES/DATA say do not shift it by a constant; the rest of the pipeline still runs.)")
tag_raw = pd.DataFrame([{"code": t["code"], "s": t["start_frame"], "e": t["end_frame"], **t["tags"]} for t in TAGS])
print(tag_raw.code.value_counts().to_string())
print("Team tag values:", tag_raw["Team"].value_counts().to_dict())

# ---- Homography ----
HMAT, HERR, HVALID = load_homography.load()
print("\nhomography: rows %d (== N: %s), valid %.1f%%, median reprojection error during play %.3f"
      % (len(HVALID), len(HVALID) == N, 100 * HVALID.mean(), np.median(HERR[GRADED])))

# ---- where did all of this come from? (Drive vs scratch) ----
print("\nread from:")
for key in ("atd_meta", "atd_raw", "homography", "smart_tagging"):
    print("  %-14s %s" % (key, PATHS[key]))
if ATD_PARSE_PATH != ATD_DRIVE_PATH:
    print("  %-14s %s  (scratch copy of atd_raw)" % ("parsed text", ATD_PARSE_PATH))

In [ ]:
if HAVE_PLT and len(vis_blocks):
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
    for h, g in vis_blocks.groupby("half"):
        ax[0].plot(g.minute, g["ball_visible_%"], marker="o", label="half %d" % h)
    ax[0].axhline(100 * vis_graded, ls="--", c="grey"); ax[0].set_ylim(0, 100)
    ax[0].set_title("ball visible %% per 5-min block (public window = first point)")
    ax[0].set_xlabel("minute in half"); ax[0].legend()
    if len(gaps):
        ax[1].hist(gaps / FPS, bins=40); ax[1].set_title("ball-missing gap lengths (s)"); ax[1].set_xlabel("seconds")
    plt.tight_layout(); plt.show()
else:
    print("plot skipped (matplotlib unavailable)")

## 3 · Processing: from tracking + tags to a possession label per frame

### Design (and why)
The two facts from the visibility block in section 2 drive everything: **the ball is missing on more than 40% of graded frames, and the gaps are long rather than scattered  section 2 measures both on this bundle**, and **every stoppage splits a possession**.
So the method is a small probabilistic decoder over *evidence*, not a nearest-player vote:

1. **Dead ball vs alive**  decoded first, because a dead stretch ends a possession.
   Evidence: ball outside the pitch; ball stationary ≥ 3 s; **a slow ball that no player comes within 3 m of for a second** (nobody is playing it  the cue that catches a celebration or a slow restart, where the ball is neither still nor out); a **player-only model** (learned *without any ground truth*, see below) for frames with no ball;
   Smart-Tagging set-piece and goal windows, **enforced** as dead until the ball moves rather than merely voted for. Rules from the docs: the first 2,934 frames of half 2 are `D`; half 1 is graded from kickoff.
2. **Team inside each alive stretch**  a 2-state Viterbi over log-odds evidence with a **switch penalty** (in nat·seconds). A flip needs *accumulated* evidence, which is the
   "controlled possession" idea from CHALLENGE.md: a loose ball or a deflection leaves possession where it was.
   Evidence: `κ·(d_away − d_home)` from the ball (gated to 0 when the ball is far from everyone, i.e. in flight); the **team that last had a foot on the ball** (within reach for at least 0.12 s, so a single frame of proximity is not a touch), carried as a state through the frames where the ball is in flight or missing; Smart-Tagging phases with the correct *possession semantics*
   (Build Up/Progression/… ⇒ that team has it; **Defending…/Defensive Transition ⇒ the *opponent* has it**); and a **team-shape model** for ball-less frames.
3. **Self-supervised player-only models.** On frames where the ball *is* visible we derive confident pseudo-labels from ball kinematics/proximity, train two small logistic models that see
   **only players** (team centroids/spreads/speeds/advance in attacking direction), and apply them where the ball is missing. No ground truth, public or private, is used.

Every threshold lives in `CFG` below. They are set from football/physics priors, with two exceptions that only labels can set - the dead-ball displacement cue and the tag weight - both selected on the public window and both disclosed in §4 (choosing the *resolution* there would still be a mistake, for the reason §4 gives).

In [ ]:
CFG = dict(
    # ball -> team evidence
    kappa=0.8, clip=4.0, r_gate=7.0,
    # smart tagging
    # The term is a weighted mean in [-1, 1], so w_tag alone bounds it. 0.75 rather than 0.5: with the
    # ball missing the tags are the only independent evidence there is, and at 0.5 the decoder ignored
    # them on the labelled window (removing the team tags changed 0 frames). Selected on that window;
    # section 4 reports what it buys and what that choice costs in independence.
    w_tag=0.75,
    # player-only team model
    w_shape=0.6, shape_clip=2.0,
    # last touch: possession carries with the team that last had a foot on the ball
    w_touch=0.5, touch_dist=1.5,
    touch_hold_s=0.12,      # the ball must stay within reach this long to count as a touch
    # ... but not forever: a touch is evidence about the *current* possession only while the ball has
    # been out of sight for a moment. On this match 67% of ball-less time sits in gaps longer than 20 s,
    # and a state that is 20 s old was outvoting the players who are actually on screen.
    touch_decay_s=5.0,
    # decoder
    lam_team=2.5,                  # nat*s needed to flip possession (higher = fewer, longer possessions)
    # Semi-Markov switch penalty. A flat lam_team says a possession is as likely to end in its
    # first second as in its fifteenth; the labels say otherwise, and two of the six possessions
    # this window misses are one long spell where there are two. h(age) below 1 makes a settled
    # possession cheaper to leave. gamma sits on a plateau - 0.20, 0.25 and 0.30 give identical
    # labels - and is taken at its midpoint rather than at the score maximum. T_s is the age at
    # which the subsidy starts; the penalty bottoms out at 1 - gamma from 2*T_s on.
    haz_gamma=0.25, haz_T_s=12.0, haz_iters=3,
    # The switch base is ensembled, not trusted on its own: three decodes at +/-10% around whatever
    # lam_team the calibration picks, combined per frame by median. A single base decides where a
    # possession ends in the ball-less stretches, where the evidence is weakest; the median is stable
    # under that choice. Measured: +0.005 masked spellF1, the same sign on three independent mask
    # sets (t 2.9-3.4), 0.000 on the clean window, 40 full-match frames changed.
    lam_ens=(0.90, 1.00, 1.10),
    lam_dead=3.0,                  # nat*s needed to enter/leave dead ball
    min_dead_s=2.0, min_alive_s=5.0,
    # dead-ball evidence
    v_static=1.0, static_min_s=3.0, oob_margin=1.0, oob_min_s=0.5, v_alive=3.0,
    # 2.0 m, not 3.0. 2.0 m is still beyond the control radius (ctrl_dist, 1.5 m), so a foot
    # cannot be on the ball, and it catches the ball resting a body-length from everyone during a
    # slow restart, which 3.0 m missed. A window-selected constant: the cue fires on 4.3 points
    # more graded frames, dead recall 70.5% -> 74.8% with precision 92.1% -> 92.5%. Section 6
    # lists it with the other values chosen by looking at the labels.
    r_iso=2.0, iso_min_s=1.0,
    # no progress: the ball is being knocked around a stoppage rather than taken anywhere
    r_np=10.0, np_win_s=3.0, np_min_s=1.0, v_np_lo=1.0, v_np_hi=8.0, kin_np=1.0,
    kin_dead=4.0, kin_alive=3.0, w_pm=0.5, pm_clip=3.0, w_sp=2.0, v_restart=3.0, sp_fallback=0.5,
    # pseudo-labels for self-supervision
    ctrl_dist=1.5, ctrl_gap=2.5,
)

# ---------------- small numeric helpers ----------------
def keep_runs(mask, min_len):
    out = np.zeros(len(mask), bool); s, e = run_bounds(mask)
    for a, b in zip(s, e):
        if b - a >= min_len:
            out[a:b] = True
    return out

def enforce_min_runs(dead, min_dead, min_alive):
    # drop dead runs shorter than min_dead, then fill alive gaps shorter than min_alive between dead runs
    dead = dead.copy(); s, e = run_bounds(dead)
    for a, b in zip(s, e):
        if b - a < min_dead:
            dead[a:b] = False
    s, e = run_bounds(~dead)
    for a, b in zip(s, e):
        if b - a < min_alive and a > 0 and b < len(dead):
            dead[a:b] = True
    return dead

def moving_mean(x, half):
    m = ~np.isnan(x); v = np.where(m, x, 0.0)
    cs = np.concatenate(([0.0], np.cumsum(v))); cm = np.concatenate(([0.0], np.cumsum(m)))
    idx = np.arange(len(x)); lo = np.maximum(0, idx - half); hi = np.minimum(len(x), idx + half + 1)
    num, den = cs[hi] - cs[lo], cm[hi] - cm[lo]
    return np.where(den > 0, num / np.maximum(den, 1), np.nan)

def shift_diff(x, k):
    out = np.full(len(x), np.nan); out[k:-k] = x[2*k:] - x[:-2*k]; return out

def viterbi2(score0, score1, penalty):
    # 2-state Viterbi; score* = per-frame score of each state (higher is better); switching costs `penalty`
    n = len(score0)
    if n == 0:
        return np.zeros(0, np.int8)
    back = np.zeros((n, 2), np.int8); s0, s1 = float(score0[0]), float(score1[0])
    for t in range(1, n):
        b0 = s1 - penalty
        if b0 > s0:
            back[t, 0] = 1; n0 = b0
        else:
            n0 = s0
        b1 = s0 - penalty
        if b1 > s1:
            back[t, 1] = 0; n1 = b1
        else:
            back[t, 1] = 1; n1 = s1
        s0 = n0 + score0[t]; s1 = n1 + score1[t]
    st = np.empty(n, np.int8); st[-1] = 0 if s0 >= s1 else 1
    for t in range(n - 1, 0, -1):
        st[t - 1] = back[t, st[t]]
    return st

def fit_logreg(X, y, l2=3.0, iters=40):
    n, d = X.shape; Xb = np.hstack([np.ones((n, 1)), X]); w = np.zeros(d + 1)
    R = np.eye(d + 1) * l2; R[0, 0] = 0
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(Xb @ w, -30, 30)))
        g = Xb.T @ (p - y) + R @ w
        Hs = (Xb * (p * (1 - p))[:, None]).T @ Xb + R + 1e-6 * np.eye(d + 1)
        step = np.linalg.solve(Hs, g); w -= step
        if np.max(np.abs(step)) < 1e-6:
            break
    return w

def predict_logreg(w, X):
    return 1 / (1 + np.exp(-np.clip(w[0] + X @ w[1:], -30, 30)))

def auc(score, y):
    r = pd.Series(score).rank().values; n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")

def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

### 3.1 Attack direction
The ATD is in absolute pitch coordinates, but "advancing" only means something relative to the direction a team is attacking. We read it from the team centroids at each kickoff
(no assumption about which way Barcelona kicks), and cross-check that the second half is the mirror of the first.

In [ ]:
def home_centroid_x(frames):
    sel = TEAM_OF_SLOT == 0
    values = PX[frames][:, sel] / L_M
    return float(np.nanmean(values)) if np.isfinite(values).any() else float("nan")

def kickoff_centroid(frame):
    # widen the window if the tracker has not re-acquired anybody in the first 2 s of the kickoff
    for span in (50, 250, 1000):
        value = home_centroid_x(slice(frame, min(N, frame + span)))
        if np.isfinite(value):
            return value
    return float("nan")

c1, c2 = kickoff_centroid(KICK1), kickoff_centroid(KICK2)
assert np.isfinite(c1) and np.isfinite(c2), (
    "no home player is tracked around a kickoff (frames %d / %d) - cannot infer the attack direction"
    % (KICK1, KICK2))
D1, D2 = (1 if c1 < 0.5 else -1), (1 if c2 < 0.5 else -1)
print("home centroid x at kickoffs: half1 %.2f -> attacks %sx | half2 %.2f -> attacks %sx"
      % (c1, "+" if D1 > 0 else "-", c2, "+" if D2 > 0 else "-"))
print("sides swap at half time:", D2 == -D1)
print("sanity (home centroid averaged over each half):", np.round([home_centroid_x(slice(a, b + 1)) for _, a, b in BLOCKS], 3))
assert D2 == -D1, "directions do not mirror - inspect kickoff frames"
DIRV = np.where(np.arange(N) < (BLOCKS[0][2] + BLOCKS[1][1]) // 2, D1, D2).astype(np.int8)

### 3.2 Ball-dependent features and self-supervised pseudo-labels
Everything that depends on the ball lives in one function so the §4 stress test can re-run it with a degraded ball.

In [ ]:
def ball_features(BXm, BYm, rows=None):
    n = len(BXm); lo, hi = (0, n) if rows is None else rows
    a, b = max(0, lo - 100), min(n, hi + 100)
    out = {k: np.full(n, np.nan) for k in ("dH", "dA", "bspeed")}
    bx, by, px, py = BXm[a:b], BYm[a:b], PX[a:b], PY[a:b]
    for key, tm in (("dH", 0), ("dA", 1)):
        sel = TEAM_OF_SLOT == tm
        d = np.sqrt((px[:, sel] - bx[:, None])**2 + (py[:, sel] - by[:, None])**2)
        d = np.where(np.isnan(d), np.inf, d).min(1); d[~np.isfinite(d)] = np.nan
        out[key][a:b] = d
    h = 12                                              # speed from +-0.5 s displacement: robust to tracker jitter
    sp = np.full(b - a, np.nan)
    if b - a > 2 * h:
        sp[h:-h] = np.hypot(bx[2*h:] - bx[:-2*h], by[2*h:] - by[:-2*h]) / (2 * h / FPS)
    out["bspeed"][a:b] = sp
    out["vis"] = ~np.isnan(BXm); out["BX"], out["BY"] = BXm, BYm
    out["dmin"] = np.fmin(out["dH"], out["dA"])
    return out

def pseudo_labels(bf, cfg):
    m = cfg["oob_margin"]
    oob = bf["vis"] & ((bf["BX"] < -m) | (bf["BX"] > L_M + m) | (bf["BY"] < -m) | (bf["BY"] > W_M + m))
    oob_run = keep_runs(oob, int(cfg["oob_min_s"] * FPS))
    still = np.isfinite(bf["bspeed"]) & (bf["bspeed"] < cfg["v_static"])
    still_run = keep_runs(still, int(cfg["static_min_s"] * FPS))
    dead_ps = oob_run | still_run
    alive_ps = np.isfinite(bf["bspeed"]) & (bf["bspeed"] > cfg["v_alive"]) & ~oob & ~dead_ps
    dH, dA = np.nan_to_num(bf["dH"], nan=99.0), np.nan_to_num(bf["dA"], nan=99.0)
    ctrl = bf["vis"] & ~dead_ps & (np.minimum(dH, dA) < cfg["ctrl_dist"]) & (np.abs(dA - dH) > cfg["ctrl_gap"])
    return dict(oob=oob, dead_ps=dead_ps, alive_ps=alive_ps, ctrl=ctrl, ctrl_home=(dH < dA))

BF_FULL = ball_features(BX, BY)
PS_FULL = pseudo_labels(BF_FULL, CFG)
g = GRADED
print("pseudo-labels on graded frames: dead %d | alive(fast ball) %d | controlled-by-a-team %d (%.1f%% of graded)"
      % (PS_FULL["dead_ps"][g].sum(), PS_FULL["alive_ps"][g].sum(), PS_FULL["ctrl"][g].sum(),
         100 * PS_FULL["ctrl"][g].mean()))

# QC against the public GT (window only; evaluation, not training)
w = slice(WIN_LO, WIN_HI); G = gt[w]
dp, ap, cl = PS_FULL["dead_ps"][w], PS_FULL["alive_ps"][w], PS_FULL["ctrl"][w]
print("\npseudo-label quality on the public window (a correctness check, not a tuning target):")
if dp.any():
    print("  dead pseudo-label : precision %.2f  recall of GT-D %.2f   (n=%d)"
          % ((G[dp] == pm.D).mean(), dp[G == pm.D].mean(), dp.sum()))
if ap.any():
    print("  fast-ball 'alive' : precision %.2f                                 (n=%d)"
          % ((G[ap] != pm.D).mean(), ap.sum()))
ch = PS_FULL["ctrl_home"][w]
ok = cl & (G != pm.D)
if ok.any():
    print("  controlled-team   : team agrees with GT on %.2f of %d frames"
          % ((ch[ok] == (G[ok] == pm.H)).mean(), ok.sum()))

### 3.3 Player-only features and the two self-supervised models
Features use players only  never the ball  so they remain defined in the frames where the ball is missing (section 2).
Quality is reported by **time-blocked cross-validation against the pseudo-labels** (AUC). This tells us whether the player-only signal exists at all; it is *not* an accuracy estimate for the final labels.

In [ ]:
def build_player_features():
    Xn, Yn = PX / L_M, PY / W_M
    Xo = np.where((DIRV == -1)[:, None], 1 - Xn, Xn)           # oriented so HOME always attacks +x
    V = np.clip(PS, 0, 12); f = {}
    for nm, tm in (("H", 0), ("A", 1)):
        s = TEAM_OF_SLOT == tm
        f["cx_" + nm] = np.nanmean(Xo[:, s], 1); f["cy_" + nm] = np.nanmean(Yn[:, s], 1)
        f["sx_" + nm] = np.nanstd(Xo[:, s], 1);  f["sy_" + nm] = np.nanstd(Yn[:, s], 1)
        f["v_" + nm] = np.nanmean(V[:, s], 1)
    f["ax"], f["ay"] = np.nanmean(Xo, 1), np.nanmean(Yn, 1)
    f["ssx"], f["ssy"] = np.nanstd(Xo, 1), np.nanstd(Yn, 1)
    f["v_all"], f["v_med"] = np.nanmean(V, 1), np.nanmedian(V, 1)
    f["slow"] = np.nanmean(np.where(np.isnan(V), np.nan, (V < 1.0).astype(float)), 1)
    f["ntrk"] = (~np.isnan(PX)).sum(1).astype(float)
    return f

PF = build_player_features()

def dead_matrix():
    cols = []
    for k in ("v_all", "slow", "v_med"):
        for hw in (12, 50, 125):
            cols.append(moving_mean(PF[k], hw))
    for k in ("ssx", "ssy", "ntrk"):
        cols.append(moving_mean(PF[k], 25))
    cols += [np.abs(moving_mean(PF["ax"], 25) - 0.5), np.abs(moving_mean(PF["ay"], 25) - 0.5)]
    return np.column_stack(cols)

def team_matrix():
    S = {k: moving_mean(v, 12) for k, v in PF.items()}
    cols = [S["cx_H"], S["cx_A"], S["cx_H"] - S["cx_A"], S["cy_H"], S["cy_A"], S["sx_H"], S["sx_A"], S["sy_H"], S["sy_A"],
            S["v_H"], S["v_A"], S["v_H"] - S["v_A"], S["ax"], S["ay"],
            shift_diff(S["cx_H"], 50), shift_diff(S["cx_A"], 50), shift_diff(S["ax"], 50)]
    return np.column_stack(cols)

def standardise(M):
    mu, sd = np.nanmean(M[GRADED], 0), np.nanstd(M[GRADED], 0) + 1e-9
    return np.nan_to_num((M - mu) / sd)

XD, XT = standardise(dead_matrix()), standardise(team_matrix())

# time blocks over graded frames for honest cross-validation
order = np.flatnonzero(GRADED); BLK = np.full(N, -1); BLK[order] = np.arange(len(order)) * 6 // len(order)
rng = np.random.RandomState(0)

def fit_selfsup(X, y_pos, y_neg, name, max_rows=80000, quiet=False, cv=True):
    lab = (y_pos | y_neg) & GRADED
    yy = y_pos.astype(float)
    n1, n0 = int((y_pos & lab).sum()), int((y_neg & lab).sum())
    if not quiet:
        print("[%s] pseudo-labelled frames: positive %d, negative %d" % (name, n1, n0))
    if min(n1, n0) < 300:
        if not quiet:
            print("[%s] too few labels -> model disabled (constant 0.5)" % name)
        return np.full(N, 0.5)
    idx = np.flatnonzero(lab)
    if len(idx) > max_rows:
        idx = rng.choice(idx, max_rows, replace=False)
    aucs = []
    for k in range(6 if cv else 0):
        tr, te = idx[BLK[idx] != k], idx[BLK[idx] == k]
        if min(yy[te].sum(), len(te) - yy[te].sum()) < 20 or min(yy[tr].sum(), len(tr) - yy[tr].sum()) < 20:
            continue
        w_ = fit_logreg(X[tr], yy[tr]); aucs.append(auc(predict_logreg(w_, X[te]), yy[te]))
    if not quiet:
        print("[%s] block-CV AUC vs pseudo-labels: %s  mean %.3f"
              % (name, np.round(aucs, 3), np.mean(aucs) if aucs else float("nan")))
    return predict_logreg(fit_logreg(X[idx], yy[idx]), X)

P_DEAD_PM = fit_selfsup(XD, PS_FULL["dead_ps"], PS_FULL["alive_ps"], "dead-ball model (players only)")
P_HOME_SHAPE = fit_selfsup(XT, PS_FULL["ctrl"] & PS_FULL["ctrl_home"], PS_FULL["ctrl"] & ~PS_FULL["ctrl_home"],
                           "team-shape model (players only)")

# applied where the ball is missing: how does it do on the public window? (evaluation only)
miss_w = ~VIS[w]; alive_g = (G != pm.D)
m1 = miss_w & alive_g
if m1.sum() > 50:
    print("\nteam-shape model on public-window frames with NO ball and GT alive (n=%d): agrees with GT on %.1f%%  (chance ~ 50%%)"
          % (m1.sum(), 100 * ((P_HOME_SHAPE[w][m1] > 0.5) == (G[m1] == pm.H)).mean()))
m2 = miss_w
if m2.sum() > 50 and (G[m2] == pm.D).any():
    print("dead-ball model on public-window frames with NO ball (n=%d): AUC vs GT dead %.3f"
          % (m2.sum(), auc(P_DEAD_PM[w][m2], (G[m2] == pm.D).astype(int))))

### 3.4 Smart Tagging → evidence
Mapping rules (from DATA.md/OVERVIEW.md; *Team = the team performing the event*):

| code | implies |
|---|---|
| Build Up, Progression, Creating Chances, Attacking Transition, Long Ball, Ball in Final Third, Ball in the Box, Players in the Box, Goals | **that team** has the ball |
| Defending in Attacking/Middle/Defensive Third, Defensive Transition | **the opponent** has the ball |
| Set Pieces | a **restart**: ball dead until it moves (Throw-ins have `Team = N/A`) |

Tags are *phases, not possessions* and are automatic (false positives/negatives)  they enter as modest evidence, never as labels.
The per-frame term is the **weighted mean** of the phases covering that frame, so overlapping phases cannot stack, and the term is bounded by `w_tag` by construction (a cap above it could never bind);
`w_tag = 0.5` nats is deliberately below what a possession needs, so a tag can tip a contested stretch but can never hold one on its own.

In [ ]:
def team_sign(txt):
    if txt is None:
        return 0
    t = txt.strip().lower()
    if t in ("", "n/a", "na"):
        return 0
    for name, sign in ((HOME_NAME, 1), (AWAY_NAME, -1)):        # the ATD metadata is the authority
        n = (name or "").strip().lower()
        if n and (n in t or t in n):
            return sign
    if "barc" in t:
        return 1                                   # DATA.md: home is Barcelona
    if "ath" in t or "atl" in t:
        return -1                                  # the supplied file spells it 'Atletic Club'
    rh = difflib.SequenceMatcher(None, t, (HOME_NAME or "").lower()).ratio()
    ra = difflib.SequenceMatcher(None, t, (AWAY_NAME or "").lower()).ratio()
    return 0 if max(rh, ra) < 0.6 or abs(rh - ra) < 0.05 else (1 if rh > ra else -1)

TAGDF = pd.DataFrame([dict(code=(t["code"] or "").strip().lower(), s=max(0, t["start_frame"]), e=min(N, t["end_frame"]),
                           team=team_sign(t["tags"].get("Team")), typ=(t["tags"].get("Type") or "").strip().lower()) for t in TAGS])
print("team mapping of tags:", TAGDF.team.value_counts().to_dict(), "(+1 home, -1 away, 0 = N/A or unmapped)")

CODE_W = {"build up": 1.0, "progression": 1.0, "creating chances": 1.0, "attacking transition": 0.8, "long ball": 0.6,
          "ball in final third": 1.0, "ball in the box": 1.0, "players in the box": 1.0, "goals": 1.0,
          "defending in attacking third": -0.8, "defending in middle third": -0.8, "defending in defensive third": -0.8,
          "defensive transition": -0.5}

def tag_team_raw(tagdf):
    # The WEIGHTED MEAN of the phases covering each frame, not their sum. Overlapping phases (a
    # Build Up inside a Progression) describe the same possession, so adding their weights lets one
    # phase stack its own evidence until it hits the cap - 28% of graded frames sat at the cap, and a
    # saturated term can outvote the ball for seconds at a time. The mean says what the tags agree
    # on, however many of them happen to be open.
    num, den = np.zeros(N), np.zeros(N)
    for r in tagdf.itertuples():
        w_ = CODE_W.get(r.code)
        if w_ is not None and r.team != 0:
            num[r.s:r.e] += w_ * r.team; den[r.s:r.e] += abs(w_)
    return np.where(den > 0, num / np.maximum(den, 1e-9), 0.0)

def set_piece_prior(tagdf, bspeed, cfg):
    pre = np.zeros(N, bool); info = []
    for r in tagdf[tagdf.code == "set pieces"].itertuples():
        if "kick off (start)" in r.typ:
            continue                                 # the half is graded from the kickoff itself
        s, e = r.s, r.e
        if e <= s:
            continue
        mv = np.flatnonzero(np.nan_to_num(bspeed[s:e], nan=0.0) > cfg["v_restart"])
        restart = s + int(mv[0]) if len(mv) else s + int(cfg["sp_fallback"] * (e - s))
        pre[s:restart] = True; info.append((r.typ, s, e, restart, len(mv) > 0))
    return pre, info

L_TAG_RAW = tag_team_raw(TAGDF)
L_TAG = CFG["w_tag"] * L_TAG_RAW      # bounded by w_tag: this is a mean, not a sum, so a cap above it is dead
SP_PRE_FULL, SP_INFO = set_piece_prior(TAGDF, BF_FULL["bspeed"], CFG)

# A goal is a stoppage by rule, and the GOALS tag is exactly that phase: it starts at the goal and ends
# one second before the next 'kick off (after goal)' tag (1986.24 s vs 1985.24 s, and the same for all
# four). So it is enforced as dead rather than left to the decoder, which otherwise reads the scoring
# team knocking the ball about during the celebration as a possession.
GOAL_DEAD = np.zeros(N, bool)
for _r in TAGDF[TAGDF.code == "goals"].itertuples():
    GOAL_DEAD[_r.s:min(N, _r.e)] = True
print("set pieces: %d restarts; ball-speed restart found for %d of them, fallback midpoint for the rest"
      % (len(SP_INFO), sum(i[4] for i in SP_INFO)))
if not TAG_TIME_BASE_OK:
    L_TAG = np.zeros(N)
    SP_PRE_FULL = np.zeros(N, bool)
    GOAL_DEAD[:] = False
    print("tag evidence disabled: the Smart Tagging clock does not match the video (see section 2.4)")

# ---- how do the tags line up with the PUBLIC ground truth? (read this - it validates the semantics above) ----
sgn = np.sign(L_TAG_RAW[w]); okm = (sgn != 0) & (G != pm.D)
if okm.sum():
    print("\ntag-implied team vs GT team on public-window frames inside team-bearing tags: %.1f%% (n=%d)"
          % (100 * ((sgn[okm] > 0) == (G[okm] == pm.H)).mean(), okm.sum()))
rows = []
for r in TAGDF[(TAGDF.code == "set pieces") & (TAGDF.e > WIN_LO) & (TAGDF.s < WIN_HI)].itertuples():
    seg_ = gt[max(r.s, WIN_LO):min(r.e, WIN_HI)]; c = np.bincount(seg_, minlength=4)
    rows.append(dict(type=r.typ, start=r.s, end=r.e, GT_H=c[0], GT_A=c[1], GT_D=c[2], GT_X=c[3]))
print("\nset-piece tags inside the public window and what the ground truth says over them:")
display(pd.DataFrame(rows) if rows else "none in window")

### 3.5 Decoder
`decode(F, …)` = dead/alive Viterbi → minimum-run clean-up → forced rules from the docs → per-stretch team Viterbi.
Frames outside the two halves are `X` (never scored). Inside the halves every frame gets a real label: `X` there would cost macroF1 and buys nothing on spellF1 beyond what the Viterbi already gives.

In [ ]:
STATIC = dict(P_DEAD_PM=P_DEAD_PM, P_HOME_SHAPE=P_HOME_SHAPE, L_TAG=L_TAG, goal_dead=GOAL_DEAD)

def last_touch_state(bf, cfg=CFG):
    # The team that last had a foot on the ball, carried forward as a state: set on every frame
    # where the ball is within touch_dist of a player, held through the frames where the ball is
    # in flight or missing - but aged while it is missing, so a 20 s old touch stops voting. This is
    # the "controlled possession" rule as a state rather than a
    # per-frame vote - a loose ball does not move possession until the other team touches it.
    # It is not a restatement of the distance term: as a decoder on its own it agrees with the
    # labels about as often as the whole pipeline does, and the two disagree in both directions,
    # so it enters as evidence.
    dH = np.nan_to_num(bf["dH"], nan=99.0); dA = np.nan_to_num(bf["dA"], nan=99.0)
    near = bf["vis"] & (np.fmin(dH, dA) < cfg["touch_dist"])
    # A touch is the ball staying within reach for a moment, not one frame of it: 4% of proximity
    # runs are a single frame, which is a tracker artefact or a deflection, and the state built
    # from one then holds the wrong team for as long as the ball is loose. Requiring the hold is
    # worth +0.02 spellF1 and +0.007 accuracy on the public window.
    hold = max(1, int(round(cfg["touch_hold_s"] * FPS)))
    keep = np.zeros(len(near), bool)
    for a, b in zip(*run_bounds(near)):
        if b - a >= hold:
            keep[a:b] = True
    idx = np.flatnonzero(keep)
    if not len(idx):
        return np.zeros(len(dH), np.float32)
    side = np.where(dH[idx] < dA[idx], 1, -1).astype(np.float32)
    pos = np.searchsorted(idx, np.arange(len(dH)), side="right") - 1
    state = np.where(pos >= 0, side[np.clip(pos, 0, len(idx) - 1)], 0.0).astype(np.float32)
    # Age the state while the ball is missing. The touch is evidence about who has the ball *now*; a
    # state carried unchanged through a 20 s occlusion is a claim about a possession that ended, and it
    # was outvoting the player-only model on exactly the frames that model exists for. The decay is a
    # no-op wherever the ball is visible (age 0), so it cannot change a ball-visible decision.
    tau = cfg.get("touch_decay_s", 0.0)
    if tau and tau > 0:
        vidx = np.flatnonzero(bf["vis"])
        if len(vidx):
            last = np.searchsorted(vidx, np.arange(len(state)), side="right") - 1
            lastf = np.where(last >= 0, vidx[np.clip(last, 0, len(vidx) - 1)], -10 ** 9)
            age = np.arange(len(state)) - lastf
            state = state * np.exp(-age / (tau * FPS)).astype(np.float32)
    return state

def make_F(BXm=None, BYm=None, rows=None, cfg=CFG, static=None):
    BXm = BX if BXm is None else BXm; BYm = BY if BYm is None else BYm
    bf = ball_features(BXm, BYm, rows); ps = pseudo_labels(bf, cfg)
    sp_pre, _ = set_piece_prior(TAGDF, bf["bspeed"], cfg)
    if not TAG_TIME_BASE_OK:
        sp_pre = np.zeros(N, bool)
    return {**(STATIC if static is None else static), **bf, **ps, "sp_pre": sp_pre,
            "last_touch": last_touch_state(bf, cfg)}

def refit_player_models(F):
    # Both player-only models are supervised by ball-derived pseudo-labels, so degrading the ball has
    # to degrade them too. The stress test used to reuse the models trained on the full ball, which
    # holds the most ball-dependent component fixed and makes the degradation look milder than it is.
    # The features are ball-free, so only the pseudo-labels and the coefficients are recomputed here.
    return {"P_DEAD_PM": fit_selfsup(XD, F["dead_ps"], F["alive_ps"], "dead-ball model (refit)",
                                     quiet=True, cv=False),
            "P_HOME_SHAPE": fit_selfsup(XT, F["ctrl"] & F["ctrl_home"], F["ctrl"] & ~F["ctrl_home"],
                                        "team-shape model (refit)", quiet=True, cv=False)}

def team_evidence(F, cfg, use_tags=True, use_shape=True, use_touch=True):
    dH, dA = np.nan_to_num(F["dH"], nan=60.0), np.nan_to_num(F["dA"], nan=60.0)
    gate = np.exp(-(np.nan_to_num(F["dmin"], nan=60.0) / cfg["r_gate"]) ** 2)
    L = np.clip(cfg["kappa"] * (dA - dH), -cfg["clip"], cfg["clip"]) * gate
    L = np.where(F["vis"], L, 0.0)
    if use_touch:
        L = L + cfg["w_touch"] * F["last_touch"]
    if use_tags:
        L = L + F["L_TAG"]
    if use_shape:
        L = L + np.where(F["vis"], 0.0, np.clip(cfg["w_shape"] * logit(F["P_HOME_SHAPE"]), -cfg["shape_clip"], cfg["shape_clip"]))
    return L

def isolated_ball(F, cfg):
    # Nobody is playing the ball: it is further than r_iso from all 46 slots and not travelling at the
    # speed of a pass, sustained long enough that a touch is ruled out.
    # r_iso is 2.0 m: beyond the 1.5 m radius at which the tracker's ball and a foot are the same
    # object (ctrl_dist, the pseudo-label control radius), so no player can plausibly be touching it,
    # and close enough to catch the ball resting a body-length away from everyone. The speed
    # cap used to be v_alive (3 m/s), which excluded a ball rolling around a stoppage - the exact case
    # this cue exists for. It is now v_np_hi (8 m/s): still below a driven pass, so a ball in flight is
    # not swallowed, but a ball ambling at 4-7 m/s with nobody near it counts. Measured: dead recall
    # 67.4% -> 70.5% on the labelled window, macroF1 0.829 -> 0.838, spellF1 unchanged.
    m = F["vis"] & (np.nan_to_num(F["bspeed"], nan=0.0) < cfg["v_np_hi"]) \
        & (np.nan_to_num(F["dmin"], nan=99.0) > cfg["r_iso"])
    return keep_runs(m, int(cfg["iso_min_s"] * FPS))

def net_displacement(F, cfg):
    # How far the ball actually gets over a centred window: |p(t+h) - p(t-h)|. Displacement, not
    # path length, is what separates a stoppage from play - a ball knocked about during a
    # celebration covers ground without going anywhere, a ball in open play crosses the pitch.
    # On the public window, 3 s of displacement ranks dead above alive with AUC 0.80, and it keeps
    # its lift inside every ball-speed band (correlation with ball speed is only 0.28), so it is
    # not a restatement of "slow ball".
    h = max(1, int(round(0.5 * cfg["np_win_s"] * FPS)))
    out = np.full(len(F["BX"]), np.nan)
    if len(out) > 2 * h:
        out[h:len(out) - h] = np.hypot(F["BX"][2 * h:] - F["BX"][:len(out) - 2 * h],
                                      F["BY"][2 * h:] - F["BY"][:len(out) - 2 * h])
    return out

def no_progress(F, cfg):
    # Moving, but not going anywhere. The speed band excludes the still-ball rule's territory
    # below it and a long pass above it; the duration requirement stops one frame of apparent
    # stillness from voting.
    sp = np.nan_to_num(F["bspeed"], nan=0.0)
    m = F["vis"] & (sp >= cfg["v_np_lo"]) & (sp < cfg["v_np_hi"]) \
        & (np.nan_to_num(net_displacement(F, cfg), nan=99.0) < cfg["r_np"])
    return keep_runs(m, int(cfg["np_min_s"] * FPS))

def dead_evidence(F, cfg, use_tags=True, use_pm=True):
    LD = np.where(F["dead_ps"], cfg["kin_dead"], 0.0) - np.where(F["alive_ps"], cfg["kin_alive"], 0.0)
    LD = LD + np.where(isolated_ball(F, cfg), cfg["kin_dead"], 0.0)
    LD = LD + np.where(no_progress(F, cfg), cfg["kin_np"], 0.0)
    if use_pm:
        LD = LD + cfg["w_pm"] * np.clip(logit(F["P_DEAD_PM"]), -cfg["pm_clip"], cfg["pm_clip"])
    if use_tags:
        LD = LD + cfg["w_sp"] * F["sp_pre"]
    return LD

def viterbi2_pen(score0, score1, pen):
    # Same two-state DP as viterbi2, except that switching into a state at frame t costs pen[t].
    # Kept separate rather than parameterising viterbi2: the dead mask uses the fixed-penalty form
    # on every call, and only the team pass needs a penalty that varies along the path.
    n = len(score0)
    if n == 0:
        return np.zeros(0, np.int8)
    back = np.zeros((n, 2), np.int8); a0, a1 = float(score0[0]), float(score1[0])
    for t in range(1, n):
        b0 = a1 - pen[t]
        if b0 > a0:
            back[t, 0] = 1; n0 = b0
        else:
            n0 = a0
        b1 = a0 - pen[t]
        if b1 > a1:
            back[t, 1] = 0; n1 = b1
        else:
            back[t, 1] = 1; n1 = a1
        a0 = n0 + score0[t]; a1 = n1 + score1[t]
    st = np.empty(n, np.int8); st[-1] = 0 if a0 >= a1 else 1
    for t in range(n - 1, 0, -1):
        st[t - 1] = back[t, st[t]]
    return st

def hazard_scale(age_s, gamma, T_s):
    # The switch penalty as a function of how long the current possession has already lasted:
    # 1.0 while it is young, falling to 1 - gamma once it is twice T_s old and staying there.
    return 1.0 - gamma * np.clip((age_s - T_s) / T_s, 0.0, 1.0)

def team_path_one(ev, cfg):
    # ev: per-frame team evidence already divided by FPS, so the per-frame scores are +-ev/2.
    # A flat penalty first - that is the fixed point's starting guess, and the whole answer when
    # haz_gamma is 0 - then re-decode with the penalty replaced by lam_team * h(age of the label
    # the previous pass produced). The age is the age of the *path*, and every pass re-decides the
    # path under it, which is what makes this a semi-Markov decoder rather than a post-hoc re-cut
    # of a memoryless one. Two or three passes are enough to stop changing the labels.
    sc0, sc1 = 0.5 * ev, -0.5 * ev
    lam, gamma = cfg["lam_team"], cfg.get("haz_gamma", 0.0)
    lab = viterbi2(sc0, sc1, lam)
    if gamma <= 0 or len(ev) < 2:
        return lab
    idx = np.arange(len(ev))
    for _ in range(max(0, int(cfg.get("haz_iters", 3)))):
        start = np.maximum.accumulate(np.where(np.r_[True, np.diff(lab) != 0], idx, 0))
        pen = lam * hazard_scale((idx - start) / FPS, gamma, cfg["haz_T_s"])
        new = viterbi2_pen(sc0, sc1, pen)
        if np.array_equal(new, lab):
            break
        lab = new
    return lab

def team_path(ev, cfg):
    # The boundary of a possession inside a ball-less stretch is decided by little evidence, so it is
    # the place where the one calibrated switch base matters most. Instead of trusting it, decode at a
    # narrow band around it and take the per-frame median: where two of the three bases agree, that is
    # the answer; where all three disagree the frame is a genuine coin flip and the median is the middle
    # path. Costs three decodes per stretch - the rest of the pipeline is unchanged.
    mult = cfg.get("lam_ens", ())
    if not mult:
        return team_path_one(ev, cfg)
    paths = np.stack([team_path_one(ev, {**cfg, "lam_team": cfg["lam_team"] * m}) for m in mult])
    return np.median(paths, axis=0).astype(np.int8)

# ---- rebound cut --------------------------------------------------------------------------------
# A shot that an opponent controls is a change of possession, whatever the sticky team path says: the
# ball left the shooter and the other team now has it. The path cannot represent that, because it is
# paying one switch cost for a whole stretch and a shot is exactly the moment the "same possession"
# assumption breaks without a stoppage. The rule is deliberately narrow - fast ball (>20 m/s),
# travelling toward the goal it is already near (within 22 m), real control by the opposing team
# within 2 s - because a wide definition cuts clearances and long balls.
# Measured on the public window: spellF1 0.6471 -> 0.6514, macroF1 0.8353 -> 0.8482, five cuts in the
# match and one inside the graded window; +0.004 with the peaks re-detected under three independent
# mask realisations. Leave-one-out: the whole effect is the 628.3 s cut (drop it and the score is
# exactly the old one), which lifts a 3.6 s ground-truth possession from IoU 0.00 to 0.41.
def rebound_split(codes, F, cfg=CFG):
    vmin = cfg.get("rebound_v", 20.0)
    near = cfg.get("rebound_near", 22.0)
    vdir = cfg.get("rebound_dir", 8.0)
    want = int(round(cfg.get("rebound_ctrl_s", 2.0) * FPS))
    reach = cfg.get("rebound_ctrl_d", 1.2)
    minf = int(round(cfg.get("rebound_min_s", 0.5) * FPS))
    sp = np.nan_to_num(F["bspeed"], nan=0.0)
    bx = np.asarray(F["BX"], float)
    k = max(1, int(round(0.2 * FPS)))
    vx = np.zeros(N)
    ok = np.isfinite(bx)
    vx[k:N - k] = np.where(ok[:N - 2 * k] & ok[2 * k:],
                           (bx[2 * k:] - bx[:N - 2 * k]) / (2 * k / FPS), 0.0)
    cand = (sp > vmin) & (np.abs(vx) >= vdir)
    cand[1:-1] &= (sp[1:-1] >= sp[:-2]) & (sp[1:-1] >= sp[2:])
    cand &= np.minimum(bx, L_M - bx) <= near
    cand &= np.sign(vx) == np.where(bx < L_M - bx, 1, -1)     # travelling toward the goal it is near
    peaks = []
    for t in np.flatnonzero(cand):
        if not peaks or t - peaks[-1] > int(2 * FPS):
            peaks.append(int(t))
    out = codes.copy()
    for t in peaks:
        ctrl = None
        for u in range(t, min(N, t + want)):
            if F["vis"][u] and np.nan_to_num(F["dmin"][u], nan=99.0) < reach:
                ctrl = u
                break
        if ctrl is None or ctrl <= 0 or codes[ctrl] not in (0, 1) or codes[ctrl] != codes[ctrl - 1]:
            continue
        s = int(codes[ctrl])
        new = 0 if np.sign(F["dA"][ctrl] - F["dH"][ctrl]) > 0 else 1
        if new == s:
            continue
        e = ctrl
        while e < N and codes[e] == s:
            e += 1
        if e - ctrl < minf:
            continue
        out[ctrl:e] = new
    return out

def decode(F, cfg=CFG, use_tags=True, use_shape=True, use_pm=True, lo=0, hi=None):
    hi = N if hi is None else hi
    L = np.nan_to_num(team_evidence(F, cfg, use_tags, use_shape))
    LD = np.nan_to_num(dead_evidence(F, cfg, use_tags, use_pm))
    codes = np.full(N, 3, np.int8)
    for a, b in RUNS:
        a2, b2 = max(a, lo), min(b, hi)
        if b2 <= a2:
            continue
        ev = LD[a2:b2] / FPS
        dead = viterbi2(-0.5 * ev, 0.5 * ev, cfg["lam_dead"]) == 1
        dead = enforce_min_runs(dead, int(cfg["min_dead_s"] * FPS), int(cfg["min_alive_s"] * FPS))
        if use_tags:
            # A tagged restart is dead by rule, not by inference: the set-piece window runs from the tag
            # to the first ball movement, and a goal covers the celebration up to the next kickoff.
            dead |= F["sp_pre"][a2:b2]
            dead |= F["goal_dead"][a2:b2]
        # rules published in DATA.md / FAQ.md
        f0, f1 = max(HALF2_START, a2), min(KICK2, b2)
        if f1 > f0:
            dead[f0 - a2:f1 - a2] = True                      # graded-but-dead run before the 2nd-half kickoff
        if a2 <= KICK1 < b2:
            dead[KICK1 - a2:min(b2, KICK1 + FPS) - a2] = False  # half 1 is graded from the kickoff itself
        out = np.where(dead, 2, 0).astype(np.int8)
        s, e = run_bounds(~dead)
        for ss, ee in zip(s, e):
            evt = L[a2 + ss:a2 + ee] / FPS
            # state 0 -> H(0), 1 -> A(1); the switch cost falls as the possession ages
            out[ss:ee] = team_path(evt, cfg)
        codes[a2:b2] = out
    if cfg.get("use_rebound", True):
        codes = rebound_split(codes, F, cfg)
    return codes

t0 = time.time()
F_FULL = make_F()

# ---- segmentation resolution, calibrated on STRUCTURE, never on score ----
# The decoder's one free scale is how readily it splits a run: the accumulated evidence needed to
# flip the team. That is what sets how many possessions the file contains, and the only labelled
# sample says 17 in five minutes, mean 14.2 s, 3.4 per minute. Both halves of that are used below,
# because spellF1 counts spells while IoU-matching rewards their length. spellF1 itself is
# deliberately not consulted: EVALUATION.md shows that ranking variants on these 17 possessions
# picks the wrong one.
#
# Note what does NOT set the resolution: `min_alive_s` floors the alive gaps in the *dead mask*,
# while the possessions come from the team Viterbi inside each alive stretch, which is unfloored.
# Short possesions are therefore an evidence question, not a representability one - the probe
# after the calibration measures exactly that.
GT_DUR = float(poss.dur_s.mean())
GT_RATE = len(poss) / ((WIN_HI - WIN_LO) / FPS / 60)

def seg_summary(codes):
    sp = pm.possession_spells(codes, RUNS)
    d = np.array([b - a for a, b, _ in sp]) / FPS
    return dict(poss=len(sp), mean_s=float(d.mean()) if len(d) else np.nan,
                rate=len(sp) / (PLAY_FRAMES / FPS / 60),
                d_share=float((codes[GRADED] == 2).mean()))

# A 0.1 step, not 0.25: the switch penalty is continuous, and a coarse grid would report whichever
# corner it happened to sample rather than the resolution that actually fits the labels.
CALIB = []
for lam in np.round(np.arange(0.4, 2.55, 0.1), 2):
    cfg_ = {**CFG, "lam_team": lam}
    c_ = decode(F_FULL, cfg_)
    s = seg_summary(c_); s["lam_team"] = lam
    w_sp = pm.possession_spells(c_[WIN_LO:WIN_HI], [(0, WIN_HI - WIN_LO)])
    s["window_poss"] = len(w_sp)
    s["window_mean_s"] = float(np.mean([b - a for a, b, _ in w_sp]) / FPS) if w_sp else np.nan
    s["matched_17"] = int(possession_report(gt, c_, thr=0.5).matched.sum())
    CALIB.append(s)
calib = pd.DataFrame(CALIB)
calib["dur_err"] = np.abs(np.log(calib.mean_s / GT_DUR))
calib["rate_err"] = np.abs(np.log(calib.rate / GT_RATE))
calib["plausible"] = calib.d_share.between(0.15, 0.45)      # a match cannot be 90% or 5% dead
if not calib.plausible.any():                                # never leave the grid empty
    calib["plausible"] = True
# How many possessions there are and how long each lasts are two halves of the same structure, and
# spellF1 counts the first while IoU-matching rewards the second. Equal weight in log space, so the
# choice cannot be dominated by whichever happens to be easier to fit.
calib["total_err"] = calib.dur_err + calib.rate_err
# The full-match targets are flat over a range of lam_team - they have to be, they are the same
# quantity measured two ways - so an arg-min resolves the choice on a difference in the fourth decimal.
# That is not reproducible: a small change elsewhere in the pipeline moves which end of the plateau wins,
# and the ends differ by 0.02-0.03 spellF1. So take the plateau explicitly (everything within 1% of the
# best structural error) and break the tie with the same structural criterion applied where the truth
# actually exists - the labelled window. spellF1 is still never consulted.
calib["win_err"] = (np.abs(np.log(calib.window_mean_s / GT_DUR))
                    + np.abs(np.log(calib.window_poss / (GT_RATE * (WIN_HI - WIN_LO) / FPS / 60))))
display(calib.round(3))
BEST = calib[calib.plausible].sort_values("total_err").iloc[0]
PLATEAU = calib[calib.plausible & (calib.total_err <= BEST.total_err * 1.01 + 1e-9)]
BEST = PLATEAU.sort_values(["win_err", "lam_team"]).iloc[0]
CFG.update(lam_team=float(BEST.lam_team))
print("plateau of equally good full-match structure: lam_team %s; the labelled window breaks the tie -> %.2f"
      % (list(np.round(PLATEAU.lam_team.values, 2)), CFG["lam_team"]))
print("calibrated segmentation: lam_team %.2f (switch penalty, nat*s)" % CFG["lam_team"])
print("  -> %d possessions, mean %.1f s (target %.1f s from the labelled window), %.2f/min (target %.2f),"
      " dead-ball share %.1f%%" % (BEST.poss, BEST.mean_s, GT_DUR, BEST.rate, GT_RATE, 100 * BEST.d_share))
print("  chosen for possession structure, never for its score; sections 4A-4F report what that costs")

# ---- which knobs can actually change the segmentation? ----
# Check the run-length floors against the switch penalty instead of assuming: an inert setting that
# looks principled is the easiest kind of tuning mistake to defend. The floors bound the dead mask;
# the possessions are emitted by the team Viterbi and can be shorter than any floor.
def _sp_len(codes):
    return np.array([b - a for a, b, _ in pm.possession_spells(codes, RUNS)]) / FPS

_NOFLOOR = decode(F_FULL, {**CFG, "min_alive_s": 0.2, "min_dead_s": 0.2})
_DEFAULT = decode(F_FULL, CFG)
SHORTEST_POSS_S = float(_sp_len(_DEFAULT).min())
FLOOR_CHANGE_PCT = 100.0 * float(np.mean(np.asarray(_NOFLOOR)[GRADED] != np.asarray(_DEFAULT)[GRADED]))
print("shortest possession emitted: %.2f s under floors of %.1f / %.1f s -> possessions are not floored"
      % (SHORTEST_POSS_S, CFG["min_alive_s"], CFG["min_dead_s"]))
print("relaxing both floors to 0.2 s changes %.1f%% of graded labels -> the floors act on the dead mask"
      " (a short alive gap inside a stoppage), not on possession length" % FLOOR_CHANGE_PCT)

# Section 4 quotes this pair, so it is measured here rather than written down: filling short alive gaps
# inside a stoppage should recover dead frames without inventing dead ones elsewhere.
_ALIVE_ONLY = decode(F_FULL, {**CFG, "min_alive_s": 0.2})
_win_gD = gt[WIN_LO:WIN_HI] == 2

def _dead_pair(c):
    dd = c[WIN_LO:WIN_HI] == 2
    return 100 * dd[_win_gD].mean(), (100 * _win_gD[dd].mean() if dd.any() else float("nan"))

FILL_RECALL_FROM, FILL_PREC_FROM = _dead_pair(_ALIVE_ONLY)
FILL_RECALL_TO, FILL_PREC_TO = _dead_pair(_DEFAULT)
if abs(FILL_RECALL_TO - FILL_RECALL_FROM) < 0.05 and abs(FILL_PREC_TO - FILL_PREC_FROM) < 0.05:
    print("alive gaps shorter than %.1f s: the floor now changes nothing on the window (recall %.1f%%, precision "
          "%.1f%%) - the enforced tag windows already cover those gaps"
          % (CFG["min_alive_s"], FILL_RECALL_TO, FILL_PREC_TO))
else:
    print("filling alive gaps shorter than %.1f s: window dead recall %.1f%% -> %.1f%%, precision %.1f%% -> %.1f%%"
          % (CFG["min_alive_s"], FILL_RECALL_FROM, FILL_RECALL_TO, FILL_PREC_FROM, FILL_PREC_TO))
ISO_SHARE = 100 * float(isolated_ball(F_FULL, CFG)[GRADED].mean())
print("isolated-ball dead cue fires on %.1f%% of graded frames" % ISO_SHARE)
NP_SHARE = 100 * float(no_progress(F_FULL, CFG)[GRADED].mean())
print("no-progress dead cue fires on %.1f%% of graded frames" % NP_SHARE)

VARIANTS = {
    "%s_full" % TEAM_NAME:      dict(use_tags=True,  use_shape=True,  use_pm=True),
    "%s_no_tags" % TEAM_NAME:   dict(use_tags=False, use_shape=True,  use_pm=True),
    "%s_ball_only" % TEAM_NAME: dict(use_tags=False, use_shape=False, use_pm=False),
}
PREDS = {name: decode(F_FULL, CFG, **kw) for name, kw in VARIANTS.items()}
print("decoded %d variants in %.0fs" % (len(PREDS), time.time() - t0))
for n, c in PREDS.items():
    sp = pm.possession_spells(c, RUNS); L_ = np.array([b - a for a, b, _ in sp]) / FPS
    print("  %-28s possessions %4d  mean %5.1fs  D share of graded %.1f%%  X on graded %d"
          % (n, len(sp), L_.mean(), 100 * (c[GRADED] == 2).mean(), (c[GRADED] == 3).sum()))

### 3.6 Write and validate the submission
`<TEAM_NAME>_full` is the submission candidate. The file is written to the repo's
`evaluation/results/` (where the scorer reads it) **and** to the Drive work folder (where you upload it
from), then checked by the same validator the organisers use. **Re-validate the exact file you upload** 
they are byte-identical, and the hash is printed so you can prove it.

In [ ]:
RES_DIR = REPO / "evaluation" / "results"; RES_DIR.mkdir(exist_ok=True)
SUB_PATH = RES_DIR / ("%s.csv" % TEAM_NAME)
DRIVE_SUB_PATH = OUT_DIR / ("%s.csv" % TEAM_NAME)

labels = np.array(list("HADX"))[PREDS["%s_full" % TEAM_NAME]]
# lineterminator pinned: the default follows the platform, and a Windows run would otherwise
# produce CRLF and a different sha256 for the same labels. The contract accepts either; "the same
# file" means the same bytes, so it is fixed here.
pd.DataFrame({"frame": np.arange(N), "label": labels}).to_csv(
    SUB_PATH, index=False, lineterminator="\n")

reason = pm.validate_submission(pm.read_label_csv(SUB_PATH), N)
assert reason is None, reason
shutil.copyfile(SUB_PATH, DRIVE_SUB_PATH)

def sha256(path, blocks=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(blocks), b""):
            h.update(block)
    return h.hexdigest()

SUB_SHA = sha256(SUB_PATH)
assert SUB_SHA == sha256(DRIVE_SUB_PATH), "the two copies are not identical"
print("written: %s (%.1f MB)" % (SUB_PATH, SUB_PATH.stat().st_size / 1e6))
print("copied : %s" % DRIVE_SUB_PATH)
print("sha256 : %s" % SUB_SHA)

validator = subprocess.run([sys.executable, str(REPO / "evaluation" / "validate_submission.py"), str(SUB_PATH)],
                           capture_output=True, text=True)
print("\n" + validator.stdout.strip())
VALIDATOR_OK = validator.returncode == 0 and "OK" in validator.stdout
if not VALIDATOR_OK:
    print(validator.stderr)
print("validator exit code:", validator.returncode)

## 4 · Evaluation  where do we stand?

What can and cannot be measured:

| evidence | covers | independent of our tuning? | how far to trust it |
|---|---|---|---|
| **A. public ground truth** | 5 min, 17 possessions | yes for the ranking: never trained on, never used to choose a variant; used to select four constants (dead-ball displacement, tag weight, isolation radius, age-hazard gamma) | *pipeline correctness only*  widest bootstrap interval ≈ 0.3 spellF1 |
| **B. stress test** | the same 5 min, with the ball masked to match the match-wide visibility | yes | best available proxy for the **full-match difficulty** |
| **C. full-match diagnostics** | all graded frames, no labels | n/a | plausibility (counts, durations, fragmentation) |
| **D. Smart-Tagging agreement** | all graded frames, automatic labels | only for the `no_tags` variants | weak independent label; the tags are noisy |
| full-match ground truth | everything | – | **private. Cannot be computed here.** |

### 4A · Public ground truth (official metric code), with bootstrap intervals

In [ ]:
ALL = {**PREDS, **BASELINES}
tabA = score_table(ALL, gt, BOOTSTRAP_N)
display(tabA.round(3))
best_base = tabA[tabA.submission.str.startswith("baseline")].iloc[0]
ours = tabA[tabA.submission == "%s_full" % TEAM_NAME].iloc[0]
print("ours_full spellF1 %.3f [%.3f, %.3f]  vs best baseline on this window (%s) %.3f [%.3f, %.3f]"
      % (ours.spellF1, ours.spell_lo, ours.spell_hi, best_base.submission, best_base.spellF1, best_base.spell_lo, best_base.spell_hi))
overlap = ours.spell_lo <= best_base.spell_hi and best_base.spell_lo <= ours.spell_hi
print("The bootstrap intervals OVERLAP -> this window cannot separate the two." if overlap else
      "The bootstrap intervals do NOT overlap -> a real gap on this window, but still only 17 possessions.")
print("Remember: smooth125 'wins' on this window and loses on the full match (EVALUATION.md). Use this table to check correctness, not to choose methods.")

In [ ]:
rep = possession_report(gt, PREDS["%s_full" % TEAM_NAME], thr=0.5)
display(rep.round(2))
print("matched %d of %d true possessions at IoU>=0.5; %d true possessions are <4 s and matched %d of them"
      % (rep.matched.sum(), len(rep), int((rep.dur_s < 4).sum()), int(rep[rep.dur_s < 4].matched.sum())))

# ---- which of the two decisions is costing the matches? ----
# Swap each decision for the truth in turn, on the window, keeping the other. This is a diagnostic,
# not a tuning step: CFG is already fixed above, and nothing below feeds back into it. It answers
# the question the wrong way round if you guess - on this window the boundaries are nearly free,
# because the truth never has the same team either side of a stoppage, while the team call inside a
# short possession is what decides whether that possession can be matched at all.
def _decode_with_truth_dead():
    L_ = np.nan_to_num(team_evidence(F_FULL, CFG))[WIN_LO:WIN_HI]
    dm = gt[WIN_LO:WIN_HI] == 2
    out = np.where(dm, 2, 0).astype(np.int8)
    for a_, b_ in zip(*run_bounds(~dm)):
        e_ = L_[a_:b_] / FPS
        out[a_:b_] = viterbi2(0.5 * e_, -0.5 * e_, CFG["lam_team"])
    c_ = np.full(N, 3, np.int8); c_[WIN_LO:WIN_HI] = out
    return c_

# "truth teams, our dead mask": keep OUR dead ball; where we call it alive take the truth's team if the
# truth names one, and keep our own label where the truth calls it dead (no team exists to copy there).
_our = PREDS["%s_full" % TEAM_NAME][WIN_LO:WIN_HI]
_teams_gt = np.full(N, 3, np.int8)
_teams_gt[WIN_LO:WIN_HI] = np.where(_our == 2, 2, np.where(gt[WIN_LO:WIN_HI] != 2, gt[WIN_LO:WIN_HI], _our))
ORACLE = pd.DataFrame([
    dict(decision="submitted", matched=int(rep.matched.sum())),
    dict(decision="truth dead mask + our teams", matched=int(possession_report(gt, _decode_with_truth_dead(), thr=0.5).matched.sum())),
    dict(decision="truth teams + our dead mask", matched=int(possession_report(gt, _teams_gt, thr=0.5).matched.sum())),
])
display(ORACLE)
print("of %d true possessions, %d are shorter than 4 s and %d of those are matched"
      % (len(rep), int((rep.dur_s < 4).sum()), int(rep[rep.dur_s < 4].matched.sum())))

# breakdown by ball visibility inside the window: the ball-less frames are where the self-supervised models matter
res = PREDS["%s_full" % TEAM_NAME][w]; vis_w = VIS[w]
rows = []
for nm, m in (("ball visible", vis_w), ("ball missing", ~vis_w)):
    al = m & (G != pm.D); dd = m & (G == pm.D)
    rows.append(dict(subset=nm, frames=int(m.sum()),
                     team_acc_on_GT_alive=float((res[al] == G[al]).mean()) if al.any() else np.nan,
                     D_recall=float((res[dd] == pm.D).mean()) if dd.any() else np.nan,
                     D_precision=float((G[m & (res == pm.D)] == pm.D).mean()) if (m & (res == pm.D)).any() else np.nan))
display(pd.DataFrame(rows).round(3))

In [ ]:
if HAVE_PLT:
    try:
        from matplotlib.colors import ListedColormap
        cm_ = ListedColormap(["#d62728", "#1f77b4", "#7f7f7f", "#ffffff"])
        rows_ = [("ground truth", gt), ("%s_full" % TEAM_NAME, PREDS["%s_full" % TEAM_NAME]),
                 ("%s_ball_only" % TEAM_NAME, PREDS["%s_ball_only" % TEAM_NAME]),
                 ("baseline_smooth125", BASELINES["baseline_smooth125"]), ("baseline_smooth50", BASELINES["baseline_smooth50"])]
        fig, ax = plt.subplots(len(rows_) + 1, 1, figsize=(13, 0.55 * len(rows_) + 1.2), sharex=True)
        for a_, (n_, c_) in zip(ax, rows_):
            a_.imshow(c_[WIN_LO:WIN_HI][None, :], aspect="auto", cmap=cm_, vmin=0, vmax=3,
                      extent=[0, (WIN_HI - WIN_LO) / FPS, 0, 1], interpolation="nearest")
            a_.set_yticks([]); a_.set_ylabel(n_, rotation=0, ha="right", va="center", fontsize=8)
        ax[-1].imshow((~VIS[WIN_LO:WIN_HI])[None, :], aspect="auto", cmap="Greys",
                      extent=[0, (WIN_HI - WIN_LO) / FPS, 0, 1], interpolation="nearest")
        ax[-1].set_yticks([]); ax[-1].set_ylabel("ball missing", rotation=0, ha="right", va="center", fontsize=8)
        ax[-1].set_xlabel("seconds in public window")
        fig.suptitle("red = H, blue = A, grey = D", fontsize=9); plt.tight_layout(); plt.show()
    except Exception as exc:
        print("plot skipped:", exc)
else:
    print("plot skipped (matplotlib unavailable)")

### 4B · Stress test: the public window is unrepresentative, so degrade it
The window is the ball-richest stretch of the match (section 2 prints both figures, and the gap lengths the masks below are drawn from). We take the public 5 minutes and **mask the ball** with gaps drawn from the *empirical* gap-length distribution of this match
until the window's visibility matches the match-wide figure, re-run the full pipeline (features → decode), and score against the real labels  on the window only, so the number is comparable across candidates.
Averaged over many random masks and applied identically to the repo's baselines.

*Limit:* real gaps cluster in crowded/aerial phases; random placement cannot reproduce that, so treat this as a better-than-nothing proxy, not a forecast.

In [ ]:
stress = None
gap_pool = gaps[gaps > 0] if len(gaps) else np.zeros(0, np.int64)
vis_target = VIS[GRADED].mean(); vis_now = VIS[WIN_LO:WIN_HI].mean()
extra_frac = max(0.0, 1 - vis_target / vis_now) if vis_now > 0 else 0.0
print("window visibility %.1f%% -> target %.1f%%: hide ~%.0f%% of visible window frames, gap lengths drawn from %d real gaps"
      % (100 * vis_now, 100 * vis_target, 100 * extra_frac, len(gap_pool)))

def make_mask(rs, lo, hi, frac):
    m = np.zeros(N, bool)
    if len(gap_pool) == 0 or frac <= 0:
        return m
    budget = frac * VIS[lo:hi].sum()
    guard = 0
    while (m[lo:hi] & VIS[lo:hi]).sum() < budget:
        g_ = int(rs.choice(gap_pool)); s_ = int(rs.randint(lo, hi))
        m[s_:min(hi, s_ + g_)] = True
        guard += 1
        if guard > 20000:                 # pathological gap pool: stop rather than spin
            break
    return m

def baseline_masked(BXm, BYm, window):
    lo, hi = max(0, WIN_LO - 200), min(N, WIN_HI + 200)
    c = nearest_team_codes(PX, PY, BXm, BYm, TEAM_OF_SLOT, rows=(lo, hi)).astype(np.int8)
    c = np.where(c == 2, -1, c).astype(np.int8)          # repo convention: NOBODY = -1 -> D
    out = np.full(N, 3, np.int8)                          # X outside the window keeps every score window-only
    smoothed = repo_baseline.smooth(c, window)             # full length: the 200-frame margin gives the window edges context
    out[lo:hi] = pm.label_codes(smoothed[lo:hi])
    return out

gt_window = gt[WIN_LO:WIN_HI]
if STRESS_DRAWS > 0 and len(gap_pool) and extra_frac > 0:
    rs = np.random.RandomState(123); acc = {}
    t0 = time.time()
    for dr in range(STRESS_DRAWS):
        msk = make_mask(rs, WIN_LO, WIN_HI, extra_frac)
        BXm, BYm = BX.copy(), BY.copy(); BXm[msk] = np.nan; BYm[msk] = np.nan
        Fm = make_F(BXm, BYm, rows=(WIN_LO, WIN_HI))
        Fm.update(refit_player_models(Fm))       # retrain the ball-supervised models on the degraded ball
        cands = {"%s_full" % TEAM_NAME: decode(Fm, CFG, True, True, True, WIN_LO, WIN_HI),
                 # the same decoder and the same calibrated switch base without this revision's two
                 # levers, under the same mask: the one row that isolates them from every other here
                 "%s_no_levers" % TEAM_NAME: decode(Fm, {**CFG, "r_iso": 3.0, "haz_gamma": 0.0},
                                               True, True, True, WIN_LO, WIN_HI),
                 "%s_no_tags" % TEAM_NAME: decode(Fm, CFG, False, True, True, WIN_LO, WIN_HI),
                 "%s_ball_only" % TEAM_NAME: decode(Fm, CFG, False, False, False, WIN_LO, WIN_HI),
                 "baseline_smooth50": baseline_masked(BXm, BYm, 50),
                 "baseline_smooth125": baseline_masked(BXm, BYm, 125)}
        for n_, c_ in cands.items():
            r_ = pm.evaluate(gt_window, c_[WIN_LO:WIN_HI])
            acc.setdefault(n_, []).append((r_["spell_f1_mean"], r_["macro_f1"], r_["n_pred_spells"]))
        print("  draw %d/%d  (%d pseudo-labelled frames left, %.0fs elapsed)"
              % (dr + 1, STRESS_DRAWS, int(((Fm["dead_ps"] | Fm["alive_ps"]) & GRADED).sum()), time.time() - t0))
    stress = pd.DataFrame({n_: dict(spellF1_mean=np.mean([x[0] for x in v]), spellF1_sd=np.std([x[0] for x in v]),
                                    macroF1_mean=np.mean([x[1] for x in v]), macroF1_sd=np.std([x[1] for x in v]),
                                    poss_mean=np.mean([x[2] for x in v])) for n_, v in acc.items()}).T
    display(stress.sort_values("spellF1_mean", ascending=False).round(3))
    print("(ground truth has 17 possessions in the window). Compare each row's drop vs its unmasked score in 4A.")
    _prev_d = (np.array([x[0] for x in acc["%s_full" % TEAM_NAME]])
               - np.array([x[0] for x in acc["%s_no_levers" % TEAM_NAME]]))
    _sd = _prev_d.std(ddof=1)
    print("paired over the same %d draws: at this calibrated switch base the two levers are worth "
          "%+.3f spellF1 (t %.2f, %d/%d draws better); section 6 gives the 60-draw version and how "
          "much it depends on that base"
          % (STRESS_DRAWS, _prev_d.mean(),
             _prev_d.mean() / (_sd / np.sqrt(len(_prev_d))) if _sd > 0 else float("nan"),
             int((_prev_d > 0).sum()), len(_prev_d)))
else:
    reason = ("MTAH_STRESS_DRAWS=0" if STRESS_DRAWS <= 0 else
              "no ball-missing gaps to draw from" if not len(gap_pool) else
              "the window is already at or below the match-wide visibility, so there is nothing to hide")
    print("stress test skipped (%s)" % reason)

### 4C · Full-match diagnostics (no labels needed)
These are plausibility checks on the *whole* graded match. `poss_per_min` is possessions per minute of *play* time (graded time minus the pre-kickoff run, which is dead by rule); the public window gives ~3.4 per minute as a rough
reference (17 possessions in 5 min  a weak prior with a wide interval, in the easiest part of the match), and the baselines show what over-fragmentation looks like (`baseline_raw`: ~1,950 possessions, 80% shorter than 3 s).

In [ ]:
def diag(c):
    sp = pm.possession_spells(c, RUNS); L_ = np.array([b - a for a, b, _ in sp]) / FPS
    gc = c[GRADED]
    return dict(possessions=len(sp), mean_s=L_.mean(), median_s=np.median(L_), p10_s=np.percentile(L_, 10), p90_s=np.percentile(L_, 90),
                share_lt3s=(L_ < 3).mean(), D_share=(gc == 2).mean(), H_share_of_alive=(gc == 0).sum() / max(1, (gc < 2).sum()),
                poss_per_min=len(sp) / (PLAY_FRAMES / FPS / 60), transitions=len(pm.transitions(c, RUNS)))
dg = pd.DataFrame({n: diag(c) for n, c in {**PREDS, **BASELINES}.items()}).T
display(dg.round(3))

# Rate and duration are not independent: every stopped second removes a possession. With our dead share the two
# labelled targets (3.4/min, 14.2 s) cannot both be hit by one knob, which is why the calibration lands between
# them instead of on either.
_o = dg.loc["%s_full" % TEAM_NAME]
print("ceiling implied by the dead-ball share: %.1f%% dead + a %.1f s mean = at most %.2f possessions per play"
      " minute (the labelled window asks 3.40 at 14.2 s, i.e. a %.0f%% dead share)"
      % (100 * _o.D_share, _o.mean_s, (1 - _o.D_share) * 60 / _o.mean_s, 100 * (1 - GT_RATE * GT_DUR / 60)))

vis_g = VIS[GRADED]; c_full = PREDS["%s_full" % TEAM_NAME]
by_vis = {}
for nm, m in (("ball visible", VIS), ("ball missing", ~VIS)):
    mm = m & GRADED
    by_vis[nm] = dict(frames=int(mm.sum()), D=(c_full[mm] == 2).mean(), H=(c_full[mm] == 0).mean(), A=(c_full[mm] == 1).mean())
display(pd.DataFrame(by_vis).T.round(3))
print("Our D share on ball-missing frames vs visible frames: a large gap would mean 'ball missing => D' leaked back in (the repo baseline's main flaw).")

### 4D · Agreement with Smart-Tagging team tags across the full match (weak, independent label)

In [ ]:
def tag_agreement(c):
    s = np.sign(L_TAG_RAW); m = GRADED & (s != 0) & (c < 2)
    pred_sign = np.where(c == 0, 1, -1)
    return float((pred_sign[m] == s[m]).mean()), int(m.sum())

rows = []
for n, c in PREDS.items():
    a_, n_ = tag_agreement(c)
    circular = "tags are an input" if n.endswith("_full") else "independent of tags"
    rows.append(dict(variant=n, agreement=a_, frames=n_, note=circular))
display(pd.DataFrame(rows).round(3))
print("Only the *_no_tags / *_ball_only rows are independent evidence. The tags are automatic, so ~100% is neither expected nor desirable.")

### 4E · Sensitivity to the one knob that matters (and why not to tune it here)
`lam_team` (nat·s needed to flip possession) trades fragmentation against missed short possessions. Shown with bootstrap intervals so you can see how little 17 possessions can say.
**Pick it with label-free evidence (4C) and football sense, not by taking the arg-max of this table**  EVALUATION.md's own baseline sweep shows that mistake.

In [ ]:
rows = []
for lam in (0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 4.0, 6.0, 10.0):
    cfg_ = {**CFG, "lam_team": lam}
    c = decode(F_FULL, cfg_)
    r = score_one(gt, c, min(300, BOOTSTRAP_N) if BOOTSTRAP_N else 0); sp = pm.possession_spells(c, RUNS)
    # structure inside the window, which is what a duration/rate mismatch shows up as
    w_sp = pm.possession_spells(c[WIN_LO:WIN_HI], [(0, WIN_HI - WIN_LO)])
    w_rep = possession_report(gt, c, thr=0.5)
    rows.append(dict(lam_team=lam, full_match_possessions=len(sp), mean_s=np.mean([b - a for a, b, _ in sp]) / FPS,
                     window_poss=len(w_sp), window_mean_s=np.mean([b - a for a, b, _ in w_sp]) / FPS if w_sp else np.nan,
                     matched_of_17=int(w_rep.matched.sum()),
                     window_spellF1=r["spell_f1_mean"], lo=r["spell_bootstrap_low"], hi=r["spell_bootstrap_high"],
                     window_macroF1=r["macro_f1"]))
display(pd.DataFrame(rows).round(3))
# ---- the levers added in this revision, measured here rather than quoted from the probes ----
# Every row is the whole pipeline re-decoded, so these are windows scores, not summed deltas. The first
# row is the submission this revision replaces, at the switch base it shipped with; the rest are at the
# base the structural calibration picks for this decoder (printed with the calibration above).
def _lever(nm, cfg_):
    c_ = decode(F_FULL, cfg_)
    r_ = score_one(gt, c_, 0)
    sp_ = pm.possession_spells(c_, RUNS)
    return dict(variant=nm, spellF1=r_["spell_f1_mean"], macroF1=r_["macro_f1"],
                matched=int(possession_report(gt, c_, thr=0.5).matched.sum()),
                poss=len(sp_), poss_per_min=len(sp_) / (PLAY_FRAMES / FPS / 60),
                dead_pct=100 * float((c_[GRADED] == 2).mean()))

_ours = PREDS["%s_full" % TEAM_NAME]
_r_ours = score_one(gt, _ours, 0)
_sp_ours = pm.possession_spells(_ours, RUNS)
LEVERS = pd.DataFrame([
    _lever("previous submission", {**CFG, "r_iso": 3.0, "haz_gamma": 0.0, "lam_team": 1.3}),
    _lever("no levers", {**CFG, "r_iso": 3.0, "haz_gamma": 0.0}),
    _lever("+ isolation radius", {**CFG, "haz_gamma": 0.0}),
    _lever("+ age hazard", {**CFG, "r_iso": 3.0}),
    dict(variant="both (submitted)", spellF1=_r_ours["spell_f1_mean"], macroF1=_r_ours["macro_f1"],
         matched=int(possession_report(gt, _ours, thr=0.5).matched.sum()),
         poss=len(_sp_ours), poss_per_min=len(_sp_ours) / (PLAY_FRAMES / FPS / 60),
         dead_pct=100 * float((_ours[GRADED] == 2).mean())),
])
display(LEVERS.round(3))
LEV_PREV_SUB, LEV_OFF, LEV_ISO, LEV_HAZ, LEV_BOTH = [LEVERS.iloc[i] for i in range(len(LEVERS))]

# The rebound cut added in this revision, isolated the same way: the same decoder, post-pass off.
REB = pd.DataFrame([_lever("without rebound", {**CFG, "use_rebound": False}),
                    _lever("with rebound (submitted)", {**CFG, "use_rebound": True})])
display(REB.round(3))
REB_OFF, REB_ON = REB.iloc[0], REB.iloc[1]

# The paired masked test (60 draws, identical masks, both player models refit per draw) runs in the probe
# harness around this notebook, not inside it: 4B can only afford ten draws. Recorded here in one place.
# Delta spellF1 against the previous submission at the calibrated switch base, and the levers on their own
# (this decoder minus the same decoder without them, at that base) - the decomposition that says which of
# the two moving parts is doing the work.
MASKED_DECOMP = pd.DataFrame([
    dict(variant="no levers, at this base", delta=-0.0176, t=-5.65, wins=3, draws=60),
    dict(variant="+ isolation radius", delta=-0.0094, t=-2.56, wins=17, draws=60),
    dict(variant="+ age hazard", delta=0.0056, t=1.55, wins=23, draws=60),
    dict(variant="both (submitted)", delta=0.0132, t=2.94, wins=37, draws=60),
])
MASKED_LEVERS_ONLY = dict(delta=0.0308, t=6.80, wins=45, draws=60)
# ... and how much all of it depends on the base: the same submitted decoder across lam_team 1.2-1.6.
MASKED_LAM = pd.DataFrame([
    dict(lam_team=1.2, both=-0.0153, t_both=-2.45, iso_only=0.0128, t_iso=3.18),
    dict(lam_team=1.3, both=-0.0226, t_both=-3.89, iso_only=0.0077, t_iso=3.63),
    dict(lam_team=1.4, both=-0.0075, t_both=-1.58, iso_only=-0.0117, t_iso=-3.52),
    dict(lam_team=1.5, both=0.0132, t_both=2.94, iso_only=-0.0094, t_iso=-2.56),
    dict(lam_team=1.6, both=0.0092, t_both=3.07, iso_only=-0.0094, t_iso=-2.59),
])
print("masked paired test, 60 draws, delta spellF1 vs the previous submission:")
display(MASKED_DECOMP.round(4))
print("levers on their own at this base: %+.4f (t %.2f, %d/%d draws better)"
      % (MASKED_LEVERS_ONLY["delta"], MASKED_LEVERS_ONLY["t"],
         MASKED_LEVERS_ONLY["wins"], MASKED_LEVERS_ONLY["draws"]))
_display_masked = MASKED_LAM.copy()
_display_masked.columns = ["lam_team", "both dF1", "t", "radius only dF1", "t"]
print("the same decoder across the switch base (submitted configuration is lam_team %.2f):" % CFG["lam_team"])
display(_display_masked.round(4))

# This round's only surviving change: the switch-base ensemble (three passes at +-10% around whatever
# base the calibration picks, combined per frame by median). Measured in the probe harness on three
# INDEPENDENT mask sets, because that is the only test where it can act.
MASKED_ENS = pd.DataFrame([
    dict(mask_seed=123, draws=100, delta=0.0055, t=3.40),
    dict(mask_seed=7, draws=60, delta=0.0042, t=2.97),
    dict(mask_seed=99, draws=60, delta=0.0059, t=3.25),
])
print("switch-base ensemble vs one base, masked paired test:")
display(MASKED_ENS.round(4))
print("pooled %+.4f over %d draws; same sign and t > 2.9 on every mask set"
      % ((MASKED_ENS.delta * MASKED_ENS.draws).sum() / MASKED_ENS.draws.sum(),
         int(MASKED_ENS.draws.sum())))

print("GT in the window: 17 possessions, mean %.1f s, 6 shorter than 4 s. Ours merges when mean_s is far above that"
      " and matched_of_17 falls  a merged possession costs both of the true ones it covers, a split one costs one"
      " false positive, which is why the metric prefers to over-split." % (poss.dur_s.mean()))

### 4F · The official scorer, run exactly as the organisers will (public ground truth)

In [ ]:
# run_evaluation.py ranks every csv in evaluation/results, so a leftover file from another team name or an
# earlier run appears on the board as a competitor. Move anything that is neither a baseline nor this run's
# file out of the way first (reversible: superseded/), and say so.
RES_DIR.mkdir(exist_ok=True)
_stale = [p for p in RES_DIR.glob("*.csv")
          if p.name != "%s.csv" % TEAM_NAME and not p.name.startswith("baseline_")]
if _stale:
    OLD_DIR = RES_DIR / "superseded"; OLD_DIR.mkdir(exist_ok=True)
    for p in _stale:
        shutil.move(str(p), str(OLD_DIR / p.name))
    print("moved %d non-submission csv(s) out of the board: %s" % (len(_stale), sorted(p.name for p in _stale)))
board = subprocess.run([sys.executable, str(REPO / "evaluation" / "run_evaluation.py"), "--bootstrap", "1000"],
                       capture_output=True, text=True, cwd=str(REPO))
print("\n".join(board.stdout.splitlines()[:26]))
if board.returncode != 0:
    print("run_evaluation.py exited", board.returncode)
    print(board.stderr[-2000:])

### 4G · Reading the results
* **Trust order:** 4F/4A confirm the pipeline is *correct* (labels aligned, file valid, metrics wired). 4B is the closest thing to a full-match forecast. 4C/4D are sanity checks.
  The ranking number  spellF1 on the private full match  is not computable by any participant.
* **Published reference points (full match):** `baseline_smooth50` spellF1 **0.2433** is the floor; `baseline_raw` 0.0486. On the public window the baselines read 0.04–0.42.
* **If 4B shows the ball-only variant collapsing but the full variant holding up,** the ball-less machinery (tags + player-only models) is doing its job; if `ours_no_tags` ≈ `ours_full`, the tags add little and you can drop that dependency.
* **Where to improve next, in expected order of value:** (1) the team evidence *inside a short possession*  the oracle rows in 4A show the boundaries are nearly free on this window while the team call decides whether a sub-4 s possession is matchable at all; (2) restart handling  dead-ball start/end timing, which the isolated-ball cue improved but did not solve (a quarter of the window's stoppage frames are still called alive, and the misses sit inside long stoppages rather than at their edges); (3) the team-shape model (richer features, e.g. pressure/line height, learned on the whole match with the same pseudo-label trick);
  (3) calibrate `κ`, `r_gate`, `lam_*` with label-free criteria or a held-out *visible-ball* proxy, never by the public score alone; (4) the broadcast ATD as a second ball source (needs frame alignment you must build);
  (5) the video itself (ball detector / homography overlay) for the long ball-less gaps.

## 5 · The deliverables `SUBMISSION.md` asks for

`SUBMISSION.md` wants three things through one form: the possession file, a **public code repository**,
and a **short technical document**  plus the rules that the labels come from an algorithm, the private
ground truth is not touched, and the repository can regenerate the exact file. Everything except the
public repository itself is produced here:

* `<TEAM_NAME>.csv`  written in §3.6, validated by the organisers' own checker, byte-identical copy on Drive
* `<TEAM_NAME>.md`  the technical document, filled in with this run's measured numbers
* `manifest.json`  inputs, config, environment, commit, hashes: enough for someone else to reproduce
* `requirements.txt`  pinned to the versions actually used

All of them land in `<Drive>/mtah2027_work/submission/`. The match data is never copied there, and
the checklist proves the bundle is not tracked by git in the repository  which is the actual rule,
not "the data must live outside the repo", since DATA.md's own instructions put it in `repo/data/`.

In [ ]:
assert "PREDS" in globals(), "run sections 0-4 first: the document and the checks use their results"

def md_table(df, floatfmt="%.3f"):
    cols = list(df.columns)
    out = ["| " + " | ".join(str(c) for c in cols) + " |",
           "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, (float, np.floating)):
                cells.append("nan" if np.isnan(v) else floatfmt % v)
            else:
                cells.append(str(v))
        out.append("| " + " | ".join(cells) + " |")
    return "\n".join(out)

score_rows = tabA[["submission", "spellF1", "macroF1", "poss", "dur_s"]].copy()
diag_full = dg.loc["%s_full" % TEAM_NAME]
label_counts = {name: int((PREDS["%s_full" % TEAM_NAME] == k).sum()) for k, name in enumerate(pm.LABELS)}

long_gap_share = (100.0 * gaps[gaps > 500].sum() / gaps.sum()) if len(gaps) else 0.0
ball_missing_share = 100.0 * (1 - VIS[GRADED].mean())
doc = []
doc.append("# %s - More than a Hack 2027, Phase 1: Ball Possession\n" % TEAM_NAME)
doc.append("Technical document generated by the submission notebook. Every number below was produced "
           "by this run, on the public five-minute ground truth; the full-match score is private.\n")
doc.append("## 1. Approach\n")
doc.append("A probabilistic decoder over evidence, not a nearest-player vote. Three layers:\n")
doc.append("1. **Dead ball vs alive**, decoded first because a stoppage ends a possession. Evidence: the ball "
           "outside the pitch, the ball stationary for 3 s or more, a ball that no player comes within %.1f m of "
           "for %.1f s at under %.1f m/s (the cue that catches a stoppage where the ball is neither still nor out - a goal celebration, "
           "a slow restart), Smart-Tagging set-piece and goal windows enforced as dead until the ball moves, and a logistic model that "
           "sees players only, for the frames where the ball is missing. A fifth cue covers the stoppage that "
           "both of those miss: over %.0f s the ball displaces less than %.0f m while moving at %.0f-%.0f m/s, so "
           "it is being knocked around rather than taken anywhere.\n"
           % (CFG["r_iso"], CFG["iso_min_s"], CFG["v_np_hi"], CFG["np_win_s"], CFG["r_np"], CFG["v_np_lo"],
              CFG["v_np_hi"]))
doc.append("2. **Team inside each alive stretch**, a two-state Viterbi over log-odds evidence with a switch "
           "penalty measured in nat-seconds, so a flip needs accumulated evidence: the 'controlled possession' "
           "rule from CHALLENGE.md, where a loose ball or a deflection leaves possession where it was. Evidence: "
           "the gated distance advantage of the ball to each team; the team that last had a foot on the ball, "
           "carried as a state through the frames where the ball is in flight or missing but aged while the ball "
           "is out of sight, so a loose ball does not move possession until somebody else touches it and a 20 s "
           "old touch stops voting; Smart-Tagging phases with possession semantics "
           "(a Defending tag means the *opponent* has the ball); and a team-shape model on ball-less frames.\n")

doc.append("The switch penalty is also not constant in time. A possession that has lasted a while is more "
           "likely to end than one that began a second ago, so the cost of leaving it falls with its age: "
           "`lambda_eff = lam_team * (1 - gamma * clip((age - T) / T, 0, 1))`, evaluated on the age of the "
           "label path the previous pass produced and re-decoded until the labels stop changing (at most "
           "three passes). At a frame, then, the penalty depends on how long the current possession has "
           "already lasted as well as on the evidence there, which is what makes this decoder semi-Markov "
           "rather than memoryless. T = %.0f s, gamma = %.2f.\n"
           % (CFG["haz_T_s"], CFG["haz_gamma"]))
doc.append("The switch base itself is not trusted on its own. The team decoder runs three passes at +/-10%% "
           "around whatever `lam_team` the structural calibration picks and takes the per-frame median, so a "
           "boundary that only one base believes in does not survive. This changes nothing on the clean "
           "window - the calibrated base is inside the band, so the median is that base - and it is the one "
           "change this round that replicated under the masked test with the same sign on three independent "
           "mask sets (%+.3f, t 2.9-3.4 pooled over %d draws), where the ball-less stretches are the only "
           "place the base has room to matter.\n"
           % ((MASKED_ENS.delta * MASKED_ENS.draws).sum() / MASKED_ENS.draws.sum(), int(MASKED_ENS.draws.sum())))
doc.append("3. **Self-supervision**, no labels involved: on frames where the ball is visible we derive confident "
           "pseudo-labels from ball kinematics and proximity, train two small L2 logistic regressions on player-only "
           "features, and apply them where the ball is missing. The public labels are used for evaluation, and to "
           "select the four constants section 6 lists - the displacement cue that finds a ball-visible "
           "stoppage, chosen on the window's 1,469 dead frames; the tag weight (0.75 instead of 0.5 nats); "
           "the isolation radius (2.0 m instead of 3.0); and the age-hazard gamma (0.25). "
           "No model is fitted to them, and the resolution is never chosen by score.\n")
doc.append("**Resolution.** The decoder's one free scale is how readily it splits a run: the accumulated evidence "
           "needed to switch team, which is what sets how many possessions the file contains. It is calibrated on "
           "two properties of the only labelled sample - the mean possession duration (%.1f s) and the rate "
           "(%.2f per minute) - and on nothing else. `spellF1` is never consulted in that choice, because ranking "
           "variants on 17 possessions picks the wrong one; section 5 reports what the chosen resolution costs.\n"
           % (GT_DUR, GT_RATE))
doc.append("The run-length floors do **not** set that resolution. `min_dead_s` drops dead stretches too "
           "short to be a stoppage; `min_alive_s` decides whether a short alive gap between two dead "
           "stretches is a separate possession or the same stoppage being restarted; and the possessions "
           "themselves come from the team decoder, unfloored - the shortest possession in the file is "
           "%.2f s. The probe above measures both: relaxing the two floors to 0.2 s changes %.1f%% of the "
           "labels - a small share, but concentrated on the stoppages that were being split in three.\n"
           % (SHORTEST_POSS_S, FLOOR_CHANGE_PCT))
doc.append("Deterministic: fixed seeds, no random initialisation, so rerunning on the same bundle reproduces the same file.\n")
doc.append("## 2. How missing and noisy tracking is handled\n")
doc.append("Every asset is automatic and uncorrected, so none of it is treated as an observation of the answer. "
           "On this bundle the ball is visible on %.1f%% of graded frames (%.1f%% across the whole video); the "
           "missing frames are not scattered, the median gap is %.1f s and %.0f%% of all ball-less graded frames sit "
           "in gaps longer than 20 s. Four consequences shape the method:\n"
           % (100 * VIS[GRADED].mean(), 100 * VIS.mean(),
              (np.median(gaps) / FPS) if len(gaps) else float("nan"), long_gap_share))
doc.append("* **No gap filling by interpolation.** Bridging a 20 s occlusion with a straight line would invent a "
           "possession. Where the ball is absent the team is decided by the player-only shape model and the Smart-Tagging "
           "prior instead, both of which stay defined.")
doc.append("* **Track slots are not players.** 46 slots for 22 people, with fragmentation and identity switches, so "
           "nothing is computed per player. Only collective features (team centroids, spreads, mean speed, advance in the "
           "attacking direction) are used, and they are mean-of-visible-values, so a slot vanishing does not become a "
           "position at (0, 0).")
doc.append("* **Noise is evidence, not truth.** Smart Tagging is automatic and carries false positives and negatives, so "
            "the tags contribute the weighted mean of the phases covering each frame - a mean, never a sum and "
            "never a label: overlapping phases average rather than stack, and `w_tag` alone bounds the term, "
            "which is what stops a tag from outvoting the ball for seconds. The ball's own position is gated by distance to the "
           "nearest player, so an isolated ball in flight contributes nothing. Ball speed is measured over +-0.5 s of "
           "displacement, which is robust to frame-to-frame tracker jitter.")
doc.append("* **The two decisions are separated.** Dead/alive is decoded first, because a stoppage ends a possession; the "
           "team is decoded only inside alive stretches. That ordering is what stops a lost ball from being read as a "
           "possession change.\n")
doc.append("## 3. Assets used, and how\n")
doc.append("| asset | used | how |\n|---|---|---|")
doc.append("| tactical ATD (players + ball) | yes | positions and speeds in metres, 46 slots, ball visibility as delivered |")
doc.append("| Smart Tagging | yes | team evidence bounded by `w_tag`, weighted mean over the phases covering each frame; set-piece and goal windows enforced dead |")
doc.append("| tactical homography | loaded and checked, not used | no decision depends on image space; it was verified one row per frame |")
doc.append("| public ground truth | evaluation, plus four constants | never an input to the decoder; used to select the no-progress displacement cue (on its 1,469 labelled dead frames), the tag weight, the isolation radius and the age-hazard gamma, and to report the scores. Section 6 lists all four together |")
doc.append("| broadcast ATD / video | dropped | a second, unsynchronised timeline with 66% homography validity; the gain did not justify a re-alignment project in the time available |")
doc.append("| own ball detection | not attempted | the honest next step for the long ball-less gaps |")
doc.append("")
doc.append("## 4. Possession rules implemented\n")
doc.append("* A possession is a maximal run of one team while the ball is alive. Every stoppage splits it, so the dead-ball "
           "decoder is as important as the team decoder.\n")
doc.append("* Possession changes only on accumulated evidence of the other team controlling the ball; a single touch, a "
           "deflection or an aerial duel that nobody controls does not flip it (the Viterbi switch penalty). Conversely possession carries with the last team to touch the ball - scored at 0.5 nats per frame whenever the ball stays within 1.5 m of a player for at least 0.12 s (one frame of proximity is a tracker artefact or a deflection, and a state built from it holds the wrong team for as long as the ball is loose) - so it takes accumulated evidence of the other team to move it.\n")
doc.append("* A restart after a set piece begins dead - enforced from the tag, not inferred - and becomes alive when "
           "the ball moves faster than %.1f m/s, or at the midpoint of the tagged window if the ball is not visible "
           "then. The four GOALS tags are enforced dead for their whole span (they end one second before the "
           "corresponding kickoff tag).\n" % CFG["v_restart"])
doc.append("* The ball is treated as dead when it is visible and no slot comes within %.1f m of it for %.1f s, at "
           "under %.1f m/s (below a driven pass, so a ball in flight is excluded): beyond the 1.5 m control "
           "radius, so no foot can be on it, and close enough to catch the ball resting a body-length from "
           "everyone. %.1f%% of graded frames satisfy it; the radius itself is one of the four constants "
           "section 6 lists as chosen on the labelled window.\n"
           % (CFG["r_iso"], CFG["iso_min_s"], CFG["v_np_hi"], ISO_SHARE))
if abs(FILL_RECALL_TO - FILL_RECALL_FROM) < 0.05 and abs(FILL_PREC_TO - FILL_PREC_FROM) < 0.05:
    doc.append("* An alive gap shorter than %.1f s between two dead stretches is the same stoppage, not a "
               "possession: the ball is being walked back or knocked about before the restart. On the labelled "
               "window the floor now moves neither dead recall nor precision (%.1f%% and %.1f%%): the tagged "
               "windows are enforced dead, so those gaps are already inside a stoppage the tag supplies. The floor "
               "remains a backstop for the stoppages no tag covers, and moves %.1f%% of graded labels.\n"
               % (CFG["min_alive_s"], FILL_RECALL_TO, FILL_PREC_TO, FLOOR_CHANGE_PCT))
else:
    doc.append("* An alive gap shorter than %.1f s between two dead stretches is the same stoppage, not a "
               "possession: the ball is being walked back or knocked about before the restart. Filling those gaps "
               "moves dead-ball recall from %.1f%% to %.1f%% on the labelled window, with precision moving from "
               "%.1f%% to %.1f%% rather than falling.\n"
               % (CFG["min_alive_s"], FILL_RECALL_FROM, FILL_RECALL_TO, FILL_PREC_FROM, FILL_PREC_TO))
doc.append("* Published forced rules: half 1 is graded from the kickoff itself; the %d graded frames of half 2 before it "
           "kicks off are dead ball.\n" % (KICK2 - HALF2_START))
doc.append("* Frames outside the two halves are written `X` (never scored). No `X` is written inside a half: it would cost "
           "macroF1 and buy nothing on spellF1 beyond what the decoder already gives.\n")
doc.append("## 5. Results on the only labelled data available\n")
doc.append("Public window (5 minutes, 17 possessions). spellF1 is the ranking metric; macroF1 breaks ties.\n")
doc.append(md_table(score_rows) + "\n")
doc.append(           "Full-match diagnostics of the submitted labels (no ground truth involved): %d possessions, mean %.1f s, "
           "median %.1f s, dead-ball share %.1f%%, %.2f possessions per play minute.\n"
           % (diag_full.possessions, diag_full.mean_s, diag_full.median_s, 100 * diag_full.D_share, diag_full.poss_per_min))
doc.append("The rate and the mean duration are not independent: every stopped second removes a possession, so a "
           "%.1f%% dead-ball share and a %.1f s mean imply about %.2f possessions per play minute, against the "
           "%.2f the file contains (a possession can survive a frame the mask calls dead, so the identity is "
           "approximate). The labelled window asks %.2f per minute at %.1f s, which is a %.0f%% dead share; no "
           "single switch penalty satisfies both, which is why the calibration lands between them.\n"
           % (100 * diag_full.D_share, diag_full.mean_s, (1 - diag_full.D_share) * 60 / diag_full.mean_s,
              diag_full.poss_per_min, GT_RATE, GT_DUR, 100 * (1 - GT_RATE * GT_DUR / 60)))
if stress is not None:
    doc.append("Stress test - the public window with the ball masked to the match-wide visibility, averaged over %d draw(s):\n" % STRESS_DRAWS)
    doc.append(md_table(stress.sort_values("spellF1_mean", ascending=False).reset_index().rename(columns={"index": "variant"})) + "\n")
doc.append("On that window the submitted labels score spellF1 **%.3f** [%.3f, %.3f] and macroF1 **%.3f**; the best "
           "shipped baseline there is %s at %.3f, and the published full-match floor is 0.2433. The intervals overlap, "
           "and this window cannot rank anything - it is the easiest five minutes of the match.\n"
           % (ours.spellF1, ours.spell_lo, ours.spell_hi, ours.macroF1, best_base.submission, best_base.spellF1))
doc.append("## 6. Limitations\n")
doc.append("* **The five choices made on the five published minutes.** RULES.md describes those minutes "
           "as a correctness check rather than a tuning set, so every value that was chosen by looking at "
           "them is listed in one place: the no-progress displacement threshold (selected on the window's "
           "1,469 labelled dead frames), the tag weight 0.75, the isolation radius 2.0 m, the age-hazard "
           "gamma 0.25, and the rebound cut's shot definition (20 m/s, within 22 m of the goal it is "
           "travelling toward, control within 2 s). Each has a mechanism behind it and none is fitted by a model, but they are "
           "window-selected values and the score they buy is part of the reported number rather than "
           "independent of it.\n")
doc.append("* **The two added in this revision, and why the switch base moved with them.** On the window the "
           "submission goes from spellF1 %.3f to %.3f (macroF1 %.3f -> %.3f, matched possessions %d -> %d of "
           "17, full-match possessions %d -> %d). The structural calibration, which never consults spellF1, "
           "moves `lam_team` %.2f -> %.2f at the same time, so the two changes have to be separated. Under the "
           "paired masked test (60 draws, identical masks, both player models refit per draw) that base move "
           "on its own is %+.3f (t %.1f) - worse than the previous submission. The two levers at that base are "
           "%+.3f (t %.1f, %d of %d draws better), which leaves the pair %+.3f net against the previous "
           "submission (t %.1f, %d of %d). So the levers are the cause and the base is the compensation they "
           "need: the age penalty makes a settled possession cheaper to leave, and a decoder that splits more "
           "readily needs a larger flat penalty so that young possessions are still protected. Neither lever "
           "works alone at this base (%+.3f for the radius, %+.3f for the hazard). It is also not robust to the "
           "base: across `lam_team` 1.2-1.6 the pair ranges %+.3f to %+.3f, and it is positive only at 1.5 and "
           "1.6 - the calibration is what puts it there, not this revision. Gamma sits on a plateau (0.20, 0.25 "
           "and 0.30 give identical labels) and is taken at its midpoint; the isolation radius does not sit on "
           "one, so it is the constant here most exposed to a 17-possession window. Expect the private match "
           "inside that range, not at the top of it.\n"
           % (LEV_PREV_SUB.spellF1, LEV_BOTH.spellF1, LEV_PREV_SUB.macroF1, LEV_BOTH.macroF1,
              int(LEV_PREV_SUB.matched), int(LEV_BOTH.matched),
              int(LEV_PREV_SUB.poss), int(LEV_BOTH.poss),
              1.3, CFG["lam_team"],
              MASKED_DECOMP.delta[0], MASKED_DECOMP.t[0],
              MASKED_LEVERS_ONLY["delta"], MASKED_LEVERS_ONLY["t"],
              MASKED_LEVERS_ONLY["wins"], MASKED_LEVERS_ONLY["draws"],
              MASKED_DECOMP.delta[3], MASKED_DECOMP.t[3], MASKED_DECOMP.wins[3], MASKED_DECOMP.draws[3],
              MASKED_DECOMP.delta[1], MASKED_DECOMP.delta[2],
              MASKED_LAM.both.min(), MASKED_LAM.both.max()))
doc.append("* **The rebound cut, and what its +0.004 rests on.** A shot that the opposing team controls "
           "is a change of possession, and the sticky path cannot represent one without a stoppage. The rule "
           "fires five times in the match and once inside the graded window; it moves the window spellF1 "
           "%.3f -> %.3f and macroF1 %.3f -> %.3f, and is positive on three independent mask realisations "
           "with the shots re-detected inside each draw. Leave-one-out says the whole effect is the 628.3 s "
           "cut, which lifts a 3.6 s ground-truth possession from IoU 0.00 to 0.41; the other four cuts fall "
           "outside the graded frames and cannot be scored. It is shipped as a rule with a measured effect, "
           "not as a calibrated constant, but on this bundle its evidence is one boundary and a second match "
           "may not contain the same shot.\n"
           % (REB_OFF.spellF1, REB_ON.spellF1, REB_OFF.macroF1, REB_ON.macroF1))
doc.append("* **Checked against two other games, and why nothing moved.** RULES.md allows human data about "
           "other games, so the touch rule was run against the two public Metrica sample matches (tracking "
           "plus their event feed; 145,006 and 141,156 frames). Their convention is last-touch and their "
           "event-derived labels are not this competition's ground truth. The ungated last-touch state "
           "agrees with them on 82.0% and 78.2% of stable frames; a ball-speed gate raises that to 94.1% "
           "and 96.9% at 8 m/s. On this window the same gate changes 1,097 frames, fixes 129 wrong team "
           "calls and breaks 332 correct ones: +0.003 at 6 m/s, -0.009 at 8 m/s, and -0.014 (t -2.1, 2 of "
           "10 draws better) under the paired mask at 6 m/s. A selective version that lets a fast touch "
           "confirm but not flip possession is 0.630 clean and -0.015 masked. Neither is shipped. The "
           "discrepancy is the clearest evidence that the team label here is not the nearest/last-touch "
           "convention the sample games use. The rebound rule was checked the same way: none of the 13 "
           "and 11 peaks detected in those games falls within 2 s of any of their 24 human SHOT events, "
           "so its shot mechanism is not externally validated; its +0.004 remains the measured "
           "one-boundary window effect above.\n")
doc.append("* **Measured this round and not shipped.** The ball-less team model at a third of its weight "
           "(`w_shape` 0.2) is +0.011 on one mask set and +0.003/+0.006 on two others, and adds nothing once "
           "the switch base is ensembled (+0.006, -0.001, -0.001), so it was left alone rather than adding a "
           "fifth tuned constant. A teacher-student refit of the same model behaves the same way (+0.013 / "
           "+0.007 / +0.005 stacked, no gain on top of the ensemble). A segmental DP with an explicit lognormal "
           "possession-length prior - the \"proper\" semi-Markov, with the duration distribution instead of "
           "the one-parameter age hazard - scores 0.429-0.471 with 6-7 of 17 spells matched: choosing cuts by "
           "accumulated evidence smears the boundaries away from where the evidence actually flips. Semantic "
           "re-weighting of the tag codes is -0.058 (dropping the defending codes) to 0.000; the crowd, "
           "ball-to-centroid and ball-inside-the-shape channels score 0.538-0.625; a look-ahead channel for "
           "passes in flight 0.624 (its sign agrees with the labels on 77.6% of alive frames, against 86% for "
           "the shipped distance channel); speed- and visibility-modulated switch penalties 0.600 clean and "
           "-0.010 masked; a duration hazard on the dead DP 0.612; per-half calibration 0.537-0.647; a spike at "
           "Smart-Tag phase boundaries changes no label at all. Two items on the earlier list cannot "
           "be built from this bundle: hand labels for this game are forbidden by the rules, and there is no "
           "second match and no video.\n")
doc.append("* The ranking number cannot be measured here: the full-match ground truth is private and the public window is "
           "the easiest five minutes of the match (the ball is visible on far more frames than average).\n")
doc.append("* Long ball-less gaps dominate the error budget. %.0f%% of graded frames have no ball, and %.0f%% of "
           "those missing frames sit in gaps longer than 20 s, where the player-only models are doing all the work "
           "on weak evidence.\n" % (ball_missing_share, long_gap_share))
_sh = rep.dur_s < 4
_ns, _nm = int(_sh.sum()), int(rep[_sh].matched.sum())
_nwrong = 0
for _r in rep[_sh].itertuples():
    _seg = PREDS["%s_full" % TEAM_NAME][int(_r.start):int(_r.start + round(_r.dur_s * FPS))]
    if (_seg < 2).any() and int(np.bincount(_seg[_seg < 2], minlength=2).argmax()) != (0 if _r.team == "H" else 1):
        _nwrong += 1
_cw = PREDS["%s_full" % TEAM_NAME][w]
_ab = (gt[w] != 2) & (_cw != 2)
_tw = _ab & (_cw != gt[w])
_dgap = np.nan_to_num(F_FULL["dH"][w] - F_FULL["dA"][w])
_tw_dist = 100 * float((np.sign(_dgap[_tw]) == np.where(gt[w][_tw] == 0, -1, 1)).mean())
_md = (gt[w] == 2) & (_cw != 2)
_md_vis = 100 * float(VIS[w][_md].mean())
_gdr, _gde = run_bounds(gt[w] == 2)
_rec = [(a_, b_, float((_cw[a_:b_] == 2).mean())) for a_, b_ in zip(_gdr, _gde)]
_worst = max(_rec, key=lambda t_: (1 - t_[2]) * (t_[1] - t_[0]))
_wl, _wh = _worst[0], _worst[1]
_wspeed = float(np.median(np.nan_to_num(F_FULL["bspeed"][w][_wl:_wh], nan=0.0)))
_wdist = float(np.median(np.nan_to_num(F_FULL["dmin"][w][_wl:_wh], nan=99.0)))
doc.append("* The remaining dead-ball error is %d frames, %.0f%% of them with the ball in view, and it is "
           "concentrated: the window's %d stoppages are found at %s. The worst is %.1f s at %.1f s, where we "
           "find %.0f%% of the dead frames; the ball is visible throughout it at a median %.1f m/s and %.1f m "
           "from the nearest player - too slow to be play, too fast for the still-ball rule.\n"
           % (int(_md.sum()), _md_vis, len(_rec),
              ", ".join("%.0f%% of %.1f s" % (100 * r[2], (r[1] - r[0]) / FPS) for r in _rec),
              (_wh - _wl) / FPS, _wl / FPS, 100 * _worst[2], _wspeed, _wdist))
doc.append("* The team call is where the rest of the headroom is, but it is a convention gap rather than a missing cue. "
           "On the %d frames we get wrong with both sides alive, the ball is nearer the team we failed to name on only "
           "%.0f%% of them; on the other %.0f%% it is nearer the team we chose. The label is therefore not a boundary in "
           "the wrong place - on those frames every geometric channel prefers the team the ground truth does not name. "
           "Two further channels were "
           "measured and did not help: velocity alignment between the ball and the nearest player (AUC 0.42 on exactly "
           "those frames). A supervised channel over thirteen geometric features, cross-validated leaving one "
           "possession out, scored 0.709 against the shipped evidence's 0.813 and lowers spellF1 whether it is "
           "added to the ball term or replaces it, so the convention is not recoverable by feature engineering "
           "on this data. What is left is the "
           "difference between 'nearest to the ball' and 'playing the ball'.\n"
           % (int(_tw.sum()), _tw_dist, 100 - _tw_dist))
doc.append("* Replacing our dead mask with the truth's and keeping our team calls still matches %d of %d possessions; "
           "replacing the team calls with the truth's and keeping our dead mask matches %d. Of the %d possessions "
           "shorter than 4 s, %d are matched, and inside %d of the rest our label is the other team. On this window the "
           "mask is therefore nearly free on spellF1 - every stoppage in it changes possession, so the team flip "
           "supplies the boundary. The case that would cost the ranking metric, a restart by the same team, is absent "
           "from these five minutes.\n"
           % (ORACLE.matched.iloc[1], len(rep), ORACLE.matched.iloc[2], _ns, _nm, _nwrong))
doc.append("* **Changes measured and rejected.** Making a tag boundary a cheap place to switch team (a per-frame switch "
           "penalty) scores 0.400 on the window and needs a larger base penalty to compensate, buying frame accuracy at the cost "
           "of possessions. Restart seeding from the set-piece tag's restarting team fires on 19 of 26 restarts and changes "
           "nothing measurable. A richer player-only dead model raises cross-validated AUC against its own pseudo-labels to "
           "0.894 but lowers the ball-less AUC that matters (0.620 to 0.600). Fitting the ball evidence on the control "
           "pseudo-labels was abandoned before implementation: that pseudo-label is defined by the distance gap the model "
           "would be given, so it scores AUC 1.000 and learns nothing. On dead ball, an asymmetric penalty that is cheaper to enter than to leave lifts the dead share without lifting dead recall; an evidence-gated run floor, which fills a short alive gap only when its own dead evidence is strong, changes no metric at all; and shortening the still-ball requirement to 2 s changes no frame (the mask is set by the transition penalty, not by those thresholds). Decaying the touch evidence with the age of the touch is shipped instead (section 1): it buys 0.000 on the clean window, where it cannot act, and 0.024 under the masked stress test. Splitting an over-long possession at its weakest evidence, snapping a boundary to the last touch, and an ensemble over mask realisations were measured and dropped as well (0.537 or worse, -0.002, and 0.000 to -0.030). Two further candidates were dropped on the stress test: a team-shape model self-trained on decoded high-confidence frames (-0.033) and a fixed 15 s dead window after each goal tag, which the tag spans make unsafe (the GOALS tags run 71-251 s). Resetting the touch state at every decoded restart changes nothing measurable: the restart's first touch overwrites it within a second, which is also why the team is wrong on half of the frames in the five seconds after a restart - the boundary, not the incumbent, is what costs there. Three candidates for the remaining team error were measured and dropped: giving the nearest player's velocity alignment with the ball a vote (AUC 0.50 over the window, 0.42 on the frames we get wrong, so it is if anything anti-correlated); a distance channel on the second-nearest player of each team (0.410-0.589 against 0.606); and a per-frame switch penalty that is cheap inside a tag boundary (0.400 - it buys frame accuracy at the cost of possessions). The age penalty was tried in the direction that punishes a young possession and is negative at every setting (0.555-0.570 against 0.606, because it suppresses the splits the labels do contain); in the direction that subsidises an old one it is shipped, and it is the one place this revision found room.\n")
doc.append("* The tracking has no identities; slots fragment and switch, so per-player reasoning is not available and "
           "collective features are all we use.\n")
doc.append("* The stress test masks the ball at random positions. Real gaps cluster in crowded and aerial phases, so it is "
           "a better-than-nothing proxy, not a forecast.\n")
doc.append("## 7. Reproduction\n")
doc.append("1. Put the match bundle in Google Drive, keeping the folder that holds `tactical/` and `broadcast/`.\n")
doc.append("2. Open the submission notebook in Colab and run all cells. It mounts Drive, finds the bundle and the "
           "repository, and regenerates the submission.\n")
doc.append("3. The submission is written to `evaluation/results/%s.csv` and copied next to this document; the run's "
           "sha256 is in `manifest.json`.\n" % TEAM_NAME)
doc.append("4. Headless equivalent, which regenerates the same file without a browser:\n")
doc.append("```bash\nMTAH_TEAM=%s MTAH_DATA_DIR='<bundle>' MTAH_WORK_DIR='<out>' \\\n  jupyter nbconvert --to notebook --execute mtah2027_possession_pipeline.ipynb --output executed.ipynb\n```\n" % TEAM_NAME)
doc.append("   The only inputs are the bundle and the public repository at commit `%s`; the notebook pins the "
           "repository, the config and the seeds, so the file it writes is byte-identical to the one submitted.\n" % REPO_REV)
doc.append("5. Dependencies: `requirements.txt` in this folder, pinned to the versions that produced this run "
           "(numpy and pandas for the pipeline and the scorer; matplotlib only for the optional plots). The "
           "public repository ships the same pins and the commands above in its `README.md`.\n")
doc.append("The match data is not redistributed with this document or the repository, per the competition terms.\n")
doc.append("Generated by the run that wrote `%s` (sha256 `%s`), so a stale copy of this document is detectable.\n"
           % (SUB_PATH.name, SUB_SHA))

TECH_DOC_PATH = OUT_DIR / ("%s.md" % TEAM_NAME)
with open(TECH_DOC_PATH, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(doc))                 # "\n", not the platform default
print("technical document:", TECH_DOC_PATH, "(%.1f kB)" % (TECH_DOC_PATH.stat().st_size / 1e3))

# Pinned to what actually produced this file. matplotlib is optional, so when it is missing the
# line is a comment rather than "matplotlib==n/a", which pip refuses to parse.
REQ_LINES = ["numpy==%s" % np.__version__, "pandas==%s" % pd.__version__]
REQ_LINES.append("matplotlib==%s" % __import__("matplotlib").__version__ if HAVE_PLT else
                 "# matplotlib (optional: only the plots in sections 1 and 4 need it)")
REQ_PATH = OUT_DIR / "requirements.txt"
with open(REQ_PATH, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(REQ_LINES) + "\n")
print("requirements    :", REQ_PATH)

In [ ]:
import platform, datetime

def sha256_or_none(path):
    try:
        return sha256(path)
    except OSError:
        return None

NOTEBOOK_SHA = None
for nb_candidate in (Path.cwd() / "mtah2027_possession_pipeline.ipynb", REPO / "mtah2027_possession_pipeline.ipynb"):
    if nb_candidate.is_file():
        NOTEBOOK_SHA = sha256_or_none(nb_candidate)
        break

def relname(path, base):
    """Relative path with forward slashes; a basename if the two are on different drives."""
    try:
        return os.path.relpath(str(path), str(base)).replace(os.sep, "/")
    except ValueError:
        return os.path.basename(str(path))

MANIFEST = dict(
    team=TEAM_NAME,
    generated_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    repository=dict(url=REPO_URL, path=Path(REPO).name, commit=REPO_REV, source=REPO_SOURCE),
    data=dict(root=Path(DATA_ROOT).name if DATA_ROOT else None, source=DATA_SOURCE, on_drive=bool(DRIVE_ROOT is not None),
              files={name: dict(path=relname(PATHS[name], DATA_ROOT), present=os.path.isfile(PATHS[name]),
                                size_bytes=os.path.getsize(PATHS[name]) if os.path.isfile(PATHS[name]) else None)
                     for name in list(NEED) + ["video"] if name in PATHS}),
    run=dict(n_frames=int(N), graded_frames=int(GRADED.sum()), fps=int(FPS), bootstrap_n=int(BOOTSTRAP_N),
             stress_draws=int(STRESS_DRAWS), config=CFG, layout=LAYOUT, parity=PARITY),
    submission=dict(path=relname(SUB_PATH, REPO), drive_copy=relname(DRIVE_SUB_PATH, OUT_DIR), sha256=SUB_SHA,
                    rows=int(len(labels)), label_counts=label_counts),
    public_window=dict(start=int(WIN_LO), stop=int(WIN_HI),
                       spellF1=float(ours.spellF1), spell_lo=float(ours.spell_lo), spell_hi=float(ours.spell_hi),
                       macroF1=float(ours.macroF1), best_baseline=best_base.submission,
                       best_baseline_spellF1=float(best_base.spellF1)),
    diagnostics=dict(possessions=int(diag_full.possessions), mean_s=float(diag_full.mean_s),
                     median_s=float(diag_full.median_s), D_share=float(diag_full.D_share),
                     poss_per_min=float(diag_full.poss_per_min)),
    environment=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
                     platform=platform.platform(), notebook_sha256=NOTEBOOK_SHA),
)
MANIFEST_PATH = OUT_DIR / "manifest.json"
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2, default=float), encoding="utf-8")
print("manifest:", MANIFEST_PATH)

### 5.1 Contract checklist

Each row is a requirement taken from `SUBMISSION.md` / `RULES.md` / `FAQ.md`, checked against the file
that was actually written  not against an intention. `INFO` rows are policy or quality notes that
cannot be asserted from a file; anything `FAIL` stops the notebook, because a submission that fails one
of these scores nothing.

In [ ]:
def data_not_published():
    """Is the bundle safe to have in the repository? DATA.md tells you to unpack into repo/data/,
    so 'inside the repository folder' is normal. RULES.md's rule is narrower and the one that
    matters: the data must not be *committed* to a public repository. Checks git, not location.
    """
    if DATA_ROOT is None:
        return False, "no data root resolved"
    root = Path(DATA_ROOT).resolve()
    if not inside(root, REPO):
        return True, "%s (outside the repository)" % root
    # Ask about the bundle's own files, not about the folder: data/README.md is deliberately
    # tracked, so `ls-files data` would report a false positive on a perfectly legal layout.
    assets = [PATHS[name] for name in list(NEED) + ["video"] if name in PATHS]
    try:
        tracked = [os.path.relpath(path, REPO) for path in assets
                   if inside(path, REPO)
                   and subprocess.run(["git", "-C", str(REPO), "ls-files", "--",
                                       os.path.relpath(path, REPO)],
                                      capture_output=True, text=True, check=False).stdout.strip()]
        probe = PATHS.get("atd_raw", str(root))
        probe = probe if inside(probe, REPO) else str(root)
        ignored = subprocess.run(["git", "-C", str(REPO), "check-ignore", "-q",
                                  os.path.relpath(probe, REPO)],
                                 capture_output=True, text=True, check=False).returncode == 0
    except OSError as exc:
        return False, "%s: git unavailable (%s), cannot prove the data is untracked" % (root, exc)
    if tracked:
        return False, "%d bundle file(s) tracked by git: %s" % (len(tracked), tracked[0])
    return True, "%s (inside the repo; %s by .gitignore, none of it tracked)" % (
        root, "ignored" if ignored else "NOT ignored - add it to .gitignore")

checks = []

def check(requirement, ok, detail, kind="FAIL"):
    # INFO is a note, not a pass: a row that cannot be asserted from the file is reported as INFO whether
    # or not its claim happens to be true, and it is not part of the hard requirement set either way.
    status = "INFO" if kind == "INFO" else ("PASS" if ok else "FAIL")
    checks.append(dict(requirement=requirement, status=status, detail=str(detail)))

table = pm.read_label_csv(SUB_PATH)
header = list(pd.read_csv(SUB_PATH, nrows=0).columns)
codes = PREDS["%s_full" % TEAM_NAME]

check("CSV named after the team", SUB_PATH.name == "%s.csv" % TEAM_NAME, SUB_PATH.name)
check("header is exactly frame,label", header == ["frame", "label"], header)
check("%d rows, frame 0..%d in order" % (N, N - 1),
      len(table) == N and np.array_equal(table["frame"].values, np.arange(N)),
      "%d rows" % len(table))
check("labels are H/A/D/X with no blank cells", pm.validate_submission(table, N) is None,
      pm.validate_submission(table, N) or "valid")
check("organisers' validate_submission.py accepts it", VALIDATOR_OK,
      "exit %s" % validator.returncode)
check("Drive copy is byte-identical to the scored file", sha256(DRIVE_SUB_PATH) == SUB_SHA, SUB_SHA[:16] + "...")
DATA_SAFE, DATA_SAFE_DETAIL = data_not_published()
check("match data is not tracked by the repository", DATA_SAFE, DATA_SAFE_DETAIL)
_doc_text = TECH_DOC_PATH.read_text(encoding="utf-8") if TECH_DOC_PATH.is_file() else ""
check("technical document is this run's, with the seven required sections",
      all(h in _doc_text for h in
          ("## 1. Approach", "## 2. How missing and noisy tracking is handled", "## 3. Assets used",
           "## 4. Possession rules", "## 5. Results", "## 6. Limitations", "## 7. Reproduction"))
      and SUB_SHA[:16] in _doc_text,
      "%s (sha %s)" % (TECH_DOC_PATH.name, SUB_SHA[:16]))
_man, _man_ok, _man_why = {}, False, "missing"
if MANIFEST_PATH.is_file():
    _man = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    _man_sha = _man.get("submission", {}).get("sha256")
    _man_t = _man.get("generated_utc")
    _csv_t = datetime.datetime.fromtimestamp(SUB_PATH.stat().st_mtime, datetime.timezone.utc)
    # A manifest from an earlier run would carry an older timestamp or another file's hash; either
    # means the deliverable folder was assembled from two runs, which is exactly what this catches.
    try:
        _man_ok = _man_sha == SUB_SHA and datetime.datetime.fromisoformat(_man_t) >= _csv_t - datetime.timedelta(seconds=2)
    except (TypeError, ValueError):
        _man_ok = False
    _man_why = "sha %s, written %s for a csv dated %s" % (str(_man_sha)[:16], _man_t, _csv_t.isoformat(timespec="seconds"))
check("reproducibility manifest is this run's, for this file", _man_ok, _man_why)
check("X only outside the graded halves", bool(np.all(codes[~GRADED] == 3)),
      "%d X on graded frames" % int((codes[GRADED] == 3).sum()))
check("no abstentions inside the graded halves", int((codes[GRADED] == 3).sum()) == 0,
      "X on a graded frame is legal but costs macroF1; the decoder never emits one")
check("parsed tracking reproduces the repo's own baseline_raw.csv", PARITY["mismatch"] <= 5,
      "%.4f%% identical, %d differing frames" % (100 * PARITY["agree"], PARITY["mismatch"]), kind="INFO")
check("labels produced by an algorithm, not by hand",
      True, "deterministic decoder: ball geometry + tags + self-supervised player models", kind="INFO")
check("full-match ground truth never read",
      True, "only the 5-minute published window is used, and only to select four constants (the no-progress "
            "threshold on its 1,469 labelled dead frames, the tag weight, the isolation radius and the "
            "age-hazard gamma, all listed in the document's section 6); the private full match is untouched",
      kind="INFO")
_t_det = time.time()
_again = decode(F_FULL, CFG)
check("deterministic across runs", bool(np.array_equal(_again, codes)),
      "a second full decode reproduced all %d frames in %.1fs" % (N, time.time() - _t_det))
check("written file matches the decoded labels",
      np.array_equal(np.array(list("HADX"))[codes], np.array(table["label"].values)),
      "%d rows compared" % len(table))
check("scored file is regenerable from repo + data",
      True, "repo %s @ %s + bundle -> sha256 %s (the doc records the same hash)" % (REPO.name, REPO_REV[:8], SUB_SHA[:16]),
      kind="INFO")

checklist = pd.DataFrame(checks)
display(checklist)
failed = checklist[checklist.status == "FAIL"]
assert failed.empty, "submission contract violated:\n" + failed.to_string(index=False)
print("\nall hard requirements satisfied; deliverables in %s" % OUT_DIR)
for path in sorted(OUT_DIR.iterdir()):
    print("  %-40s %8.1f kB" % (path.name, path.stat().st_size / 1e3))

### Submit

The form (link mailed to registered participants) asks for four things. You now have all of them:

| field | value |
|---|---|
| Team Name | `TEAM_NAME` above  must match what every member wrote at registration |
| Possession file | `<Drive>/mtah2027_work/submission/<TEAM_NAME>.csv`  upload the copy whose sha256 is printed above |
| Public code repository URL | your public repository containing this notebook, `requirements.txt` and the technical document  **never the match data** |
| Short technical document | `<Drive>/mtah2027_work/submission/<TEAM_NAME>.md` (PDF or Markdown) |

Deadline **17 November 2026**, one response per team, editable until the deadline. Before you upload,
re-open the CSV you are about to send and run `python evaluation/validate_submission.py <that file>` 
it is the single most common way a good entry scores nothing.

Keep `data/` out of the public repository (RULES.md: no redistribution, no derived exports that would
reconstruct it), and do not upload the data or extracts of it to a hosted LLM.